In [6]:
# pseudo_gt_tcn.py

from pathlib import Path
from datetime import date
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from scipy.ndimage import binary_dilation
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("/shared-docker/data/makeathon-challenge")
PROCESSED = ROOT / "processed"
CLOUD_ROOT = PROCESSED / "s2_cloudmask"

OUT = PROCESSED / "pseudo_gt_tcn"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
DATE_AGREEMENT_DAYS = 90

WINDOW_START = date(TARGET_YEARS[0], 1, 1).toordinal()
WINDOW_END   = date(TARGET_YEARS[-1] + 1, 1, 1).toordinal()

EPOCH_RADD   = date(2014, 12, 31).toordinal()
EPOCH_GLADS2 = date(2019, 1, 1).toordinal()

# negative buffer around any positive alert
NEGATIVE_BUFFER_PIXELS = 3

# source vote weights
RADD_LOW_W      = 0.35
RADD_HIGH_W     = 0.45
GLADL_PROB_W    = 0.30
GLADL_CONF_W    = 0.40
GLADS2_LOW_W    = 0.25
GLADS2_MED_W    = 0.35
GLADS2_HIGH_W   = 0.45

# date agreement bonus between source pairs
PAIR_BONUS = 0.15

# optical cloud attenuation
# if cloudy in the event month at that pixel, optical vote gets multiplied by this
OPTICAL_CLOUD_ATTENUATION = 0.50

# label thresholds
POSITIVE_THRESHOLD = 0.80
MEDIUM_THRESHOLD   = 0.60

# ignore negatives near positives
NEGATIVE_LABEL = 0.0
POSITIVE_LABEL = 1.0
IGNORE_LABEL   = -1.0


# ============================================================
# HELPERS
# ============================================================
def old_confidence_from_votes_and_dates(radd_mask, radd_date, gs2_mask, gs2_date, gl_mask, gl_date):
    votes = radd_mask.astype(np.int8) + gs2_mask.astype(np.int8) + gl_mask.astype(np.int8)

    conf_old = votes.astype(np.float32) * 0.4
    conf_old += pair_bonus(radd_mask, radd_date, gs2_mask, gs2_date)
    conf_old += pair_bonus(radd_mask, radd_date, gl_mask,  gl_date)
    conf_old += pair_bonus(gs2_mask,  gs2_date,  gl_mask,  gl_date)
    conf_old = np.clip(conf_old, 0.0, 1.0)
    return conf_old
    
def parse_year_month_from_mask_name(path: Path):
    # 18NWG_6_6__s2_l2a_2020_1_cloudmask.tif
    m = re.search(r"_(\d{4})_(\d{1,2})_cloudmask\.tif$", path.name)
    if m is None:
        raise ValueError(f"Could not parse year/month from mask name: {path.name}")
    return int(m.group(1)), int(m.group(2))


def ordinal_to_year_month(ord_arr):
    """
    Returns year_arr, month_arr for positive ordinals, else zeros.
    """
    year = np.zeros(ord_arr.shape, dtype=np.int16)
    month = np.zeros(ord_arr.shape, dtype=np.int8)

    uniq = np.unique(ord_arr[ord_arr > 0])
    for o in uniq:
        d = date.fromordinal(int(o))
        m = ord_arr == o
        year[m] = d.year
        month[m] = d.month
    return year, month


def month_index_from_year_month(year_arr, month_arr):
    """
    Jan 2020 -> 0 ... Dec 2025 -> 71, else -1
    """
    tidx = np.full(year_arr.shape, -1, dtype=np.int16)
    valid = (year_arr >= TARGET_YEARS[0]) & (year_arr <= TARGET_YEARS[-1]) & (month_arr >= 1) & (month_arr <= 12)
    tidx[valid] = ((year_arr[valid] - TARGET_YEARS[0]) * 12 + (month_arr[valid] - 1)).astype(np.int16)
    return tidx


def get_ref(tile):
    """
    Use S2 as reference grid whenever possible.
    Returns (transform, crs, shape).
    Ignores degenerate rasters.
    """
    s2_dir = ROOT / "sentinel-2" / "train" / f"{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best

    gl = sorted((ROOT / "labels" / "train" / "gladl").glob(f"gladl_{tile}_alert*.tif"))
    if gl:
        with rasterio.open(gl[0]) as r:
            return r.transform, r.crs, r.shape

    p = ROOT / "labels" / "train" / "radd" / f"radd_{tile}_labels.tif"
    if p.exists():
        with rasterio.open(p) as r:
            return r.transform, r.crs, r.shape

    return None


def reproj(path, ref, dtype):
    out = np.zeros(ref[2], dtype=dtype)
    with rasterio.open(path) as src:
        reproject(
            rasterio.band(src, 1),
            out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref[0],
            dst_crs=ref[1],
            resampling=Resampling.nearest,
        )
    return out


# ============================================================
# CLOUD MASK ACCESS
# ============================================================
def find_cloud_mask_path(tile, year, month):
    tile_dir = CLOUD_ROOT / "train" / f"{tile}__s2_l2a"
    if not tile_dir.exists():
        return None

    candidates = [
        tile_dir / f"{tile}__s2_l2a_{year}_{month}_cloudmask.tif",
        tile_dir / f"{tile}__s2_l2a_{year}_{month:02d}_cloudmask.tif",
    ]
    for c in candidates:
        if c.exists():
            return c

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month:02d}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    return None


def load_cloud_mask_on_ref(tile, year, month, ref):
    """
    Returns:
      cloudy bool raster on ref grid, or None if mask not found.
    Assumes mask encoding:
      1 = cloud
      0 = clear
    """
    p = find_cloud_mask_path(tile, year, month)
    if p is None:
        return None
    arr = reproj(p, ref, dtype=np.uint8)
    return arr == 1


def attenuate_optical_weight(base_weight, tile, year_arr, month_arr, ref):
    """
    base_weight: per-pixel float array
    For each (year, month), if cloud mask exists and pixel is cloudy, downweight.
    """
    out = base_weight.astype(np.float32).copy()
    pairs = np.unique(np.stack([year_arr, month_arr], axis=-1).reshape(-1, 2), axis=0)

    for y, m in pairs:
        y = int(y)
        m = int(m)
        if y <= 0 or m <= 0:
            continue
        cloudy = load_cloud_mask_on_ref(tile, y, m, ref)
        if cloudy is None:
            continue
        sel = (year_arr == y) & (month_arr == m) & cloudy
        out[sel] *= OPTICAL_CLOUD_ATTENUATION
    return out


# ============================================================
# LABEL LOADERS
# ============================================================
def get_radd(tile, ref):
    p = ROOT / "labels" / "train" / "radd" / f"radd_{tile}_labels.tif"
    if not p.exists():
        return None, None, None, False

    raw = reproj(p, ref, dtype=np.int32)

    conf_digit = raw // 10000
    days = raw % 10000
    date_ord = np.where(raw > 0, days + EPOCH_RADD, 0)

    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (raw > 0) & in_window

    weight = np.zeros(ref[2], dtype=np.float32)
    weight[(mask) & (conf_digit == 2)] = RADD_LOW_W
    weight[(mask) & (conf_digit == 3)] = RADD_HIGH_W

    return mask, np.where(mask, date_ord, 0), weight, True


def get_glads2(tile, ref):
    ap = ROOT / "labels" / "train" / "glads2" / f"glads2_{tile}_alert.tif"
    dp = ROOT / "labels" / "train" / "glads2" / f"glads2_{tile}_alertDate.tif"
    if not (ap.exists() and dp.exists()):
        return None, None, None, False

    alert = reproj(ap, ref, dtype=np.uint8)
    days = reproj(dp, ref, dtype=np.uint16)

    date_ord = np.where(alert > 0, days.astype(np.int64) + EPOCH_GLADS2, 0)
    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)

    # ignore "recent only" class 1
    mask = (alert >= 2) & in_window

    weight = np.zeros(ref[2], dtype=np.float32)
    weight[(mask) & (alert == 2)] = GLADS2_LOW_W
    weight[(mask) & (alert == 3)] = GLADS2_MED_W
    weight[(mask) & (alert == 4)] = GLADS2_HIGH_W

    year_arr, month_arr = ordinal_to_year_month(np.where(mask, date_ord, 0))
    weight = attenuate_optical_weight(weight, tile, year_arr, month_arr, ref)

    return mask, np.where(mask, date_ord, 0), weight, True


def get_gladl(tile, ref):
    any_file = False
    mask = np.zeros(ref[2], dtype=bool)
    date_ord = np.zeros(ref[2], dtype=np.int64)
    weight = np.zeros(ref[2], dtype=np.float32)

    for y in TARGET_YEARS:
        ap = ROOT / "labels" / "train" / "gladl" / f"gladl_{tile}_alert{y%100:02d}.tif"
        dp = ROOT / "labels" / "train" / "gladl" / f"gladl_{tile}_alertDate{y%100:02d}.tif"
        if not (ap.exists() and dp.exists()):
            continue

        any_file = True
        alert = reproj(ap, ref, dtype=np.uint8)
        doy = reproj(dp, ref, dtype=np.uint16)

        year_start = date(y, 1, 1).toordinal()
        year_date = np.where(alert > 0, doy.astype(np.int64) + year_start - 1, 0)
        in_window = (year_date >= WINDOW_START) & (year_date < WINDOW_END)
        year_mask = (alert > 0) & in_window

        # earliest date wins per pixel
        better = year_mask & ((date_ord == 0) | ((year_date > 0) & (year_date < date_ord)))
        date_ord = np.where(better, year_date, date_ord)
        mask = mask | year_mask

        year_weight = np.zeros(ref[2], dtype=np.float32)
        year_weight[(year_mask) & (alert == 2)] = GLADL_PROB_W
        year_weight[(year_mask) & (alert == 3)] = GLADL_CONF_W

        # keep the weight corresponding to the earliest kept date
        weight = np.where(better, year_weight, weight)

    if not any_file:
        return None, None, None, False

    year_arr, month_arr = ordinal_to_year_month(np.where(mask, date_ord, 0))
    weight = attenuate_optical_weight(weight, tile, year_arr, month_arr, ref)

    return mask, date_ord, weight, True


# ============================================================
# CONSENSUS / PSEUDO-GT
# ============================================================
def pair_bonus(m1, d1, m2, d2, bonus=PAIR_BONUS):
    both = m1 & m2 & (d1 > 0) & (d2 > 0)
    if not both.any():
        return np.zeros(m1.shape, dtype=np.float32)
    diff = np.abs(d1.astype(np.int64) - d2.astype(np.int64))
    close = both & (diff <= DATE_AGREEMENT_DAYS)
    return np.where(close, bonus, 0.0).astype(np.float32)


def build_negative_mask(any_positive_seed):
    """
    Negatives are allowed only away from any positive seed.
    """
    buffered = binary_dilation(any_positive_seed, structure=np.ones((2 * NEGATIVE_BUFFER_PIXELS + 1,
                                                                     2 * NEGATIVE_BUFFER_PIXELS + 1), dtype=bool))
    return ~buffered


def build_pgt(tile):
    ref = get_ref(tile)
    if ref is None:
        return None

    native_shape = ref[2]

    radd_mask, radd_date, radd_w, has_radd = get_radd(tile, ref)
    gs2_mask, gs2_date, gs2_w, has_gs2 = get_glads2(tile, ref)
    gl_mask,  gl_date,  gl_w,  has_gl  = get_gladl(tile, ref)

    if radd_mask is None:
        radd_mask = np.zeros(native_shape, dtype=bool)
        radd_date = np.zeros(native_shape, dtype=np.int64)
        radd_w = np.zeros(native_shape, dtype=np.float32)

    if gs2_mask is None:
        gs2_mask = np.zeros(native_shape, dtype=bool)
        gs2_date = np.zeros(native_shape, dtype=np.int64)
        gs2_w = np.zeros(native_shape, dtype=np.float32)

    if gl_mask is None:
        gl_mask = np.zeros(native_shape, dtype=bool)
        gl_date = np.zeros(native_shape, dtype=np.int64)
        gl_w = np.zeros(native_shape, dtype=np.float32)

    votes = radd_mask.astype(np.int8) + gs2_mask.astype(np.int8) + gl_mask.astype(np.int8)
    n_sources = int(has_radd) + int(has_gs2) + int(has_gl)

    confidence = radd_w + gs2_w + gl_w
    confidence += pair_bonus(radd_mask, radd_date, gs2_mask, gs2_date)
    confidence += pair_bonus(radd_mask, radd_date, gl_mask,  gl_date)
    confidence += pair_bonus(gs2_mask,  gs2_date,  gl_mask,  gl_date)
    confidence = np.clip(confidence, 0.0, 1.0)
    
    # confidence = old_confidence_from_votes_and_dates(
    # radd_mask, radd_date,
    # gs2_mask, gs2_date,
    # gl_mask, gl_date
    # )
    
    # positives
    strong_pos = (votes >= 2) & (confidence >= POSITIVE_THRESHOLD)
    medium_pos = (votes >= 2) & (confidence >= MEDIUM_THRESHOLD) & ~strong_pos
    pos = strong_pos | medium_pos

    # negatives only far from any alert
    any_alert = radd_mask | gs2_mask | gl_mask
    neg_allowed = build_negative_mask(pos | any_alert)
    neg = (votes == 0) & neg_allowed

    label = np.full(native_shape, IGNORE_LABEL, dtype=np.float32)
    label[pos] = POSITIVE_LABEL
    label[neg] = NEGATIVE_LABEL

    weight = np.zeros(native_shape, dtype=np.float32)
    weight[label == POSITIVE_LABEL] = confidence[label == POSITIVE_LABEL]
    weight[label == NEGATIVE_LABEL] = 1.0

    # event date for positives: median of available dates where possible
    event_ord = np.zeros(native_shape, dtype=np.int64)
    stack_dates = np.stack([radd_date, gs2_date, gl_date], axis=0)

    pos_idx = np.where(label == POSITIVE_LABEL)
    for r, c in zip(*pos_idx):
        vals = stack_dates[:, r, c]
        vals = vals[vals > 0]
        if len(vals) == 0:
            continue
        vals = np.sort(vals)
        event_ord[r, c] = vals[len(vals) // 2]

    event_year, event_month = ordinal_to_year_month(event_ord)
    event_tidx = month_index_from_year_month(event_year, event_month)

    stats = {
        "tile": tile,
        "native_shape": f"{native_shape[0]}x{native_shape[1]}",
        "n_sources": n_sources,
        "has_radd": has_radd,
        "has_gs2": has_gs2,
        "has_gl": has_gl,
        "pos_frac": float((label == POSITIVE_LABEL).mean()),
        "neg_frac": float((label == NEGATIVE_LABEL).mean()),
        "ign_frac": float((label == IGNORE_LABEL).mean()),
        "pos_mean_conf": float(weight[label == POSITIVE_LABEL].mean()) if (label == POSITIVE_LABEL).any() else 0.0,
        "strong_pos": int(strong_pos.sum()),
        "medium_pos": int(medium_pos.sum()),
    }

    for y in TARGET_YEARS:
        stats[f"pos_y{y}"] = int(((label == POSITIVE_LABEL) & (event_year == y)).sum())

    return {
        "label": label.astype(np.float32),          # 1, 0, -1
        "weight": weight.astype(np.float32),        # sample weight
        "confidence": confidence.astype(np.float32),
        "event_month": event_month.astype(np.int8), # 1..12, 0 if unknown
        "event_year": event_year.astype(np.int16),  # 2020..2025, 0 if unknown
        "event_tidx": event_tidx.astype(np.int16),  # 0..71, -1 if unknown
        "stats": stats,
    }


# ============================================================
# RUN
# ============================================================
def list_train_tiles():
    s2_train = ROOT / "sentinel-2" / "train"
    return sorted([p.name.replace("__s2_l2a", "") for p in s2_train.iterdir() if p.is_dir() and p.name.endswith("__s2_l2a")])


def main():
    stats = []

    for tile in tqdm(list_train_tiles(), desc="pseudo_gt_tcn"):
        try:
            r = build_pgt(tile)
            if r is None:
                print(f"{tile}: skipped")
                continue

            np.savez_compressed(
                OUT / f"{tile}.npz",
                label=r["label"],
                weight=r["weight"],
                confidence=r["confidence"],
                event_month=r["event_month"],
                event_year=r["event_year"],
                event_tidx=r["event_tidx"],
            )
            stats.append(r["stats"])

        except Exception as e:
            print(f"{tile}: ERROR {e}")

    df = pd.DataFrame(stats)
    df.to_csv(OUT / "stats.csv", index=False)

    pd.set_option("display.width", 300)
    pd.set_option("display.max_columns", 40)
    print("\n" + df.to_string(index=False))
    print(f"\nTotal tiles: {len(df)}")
    if len(df):
        print(f"Mean pos_frac: {df.pos_frac.mean():.3%}")
        print(f"Total strong_pos: {df.strong_pos.sum():,}")
        print(f"Total medium_pos: {df.medium_pos.sum():,}")
        print("\nPositives per year:")
        for y in TARGET_YEARS:
            print(f"  {y}: {df[f'pos_y{y}'].sum():,}")


if __name__ == "__main__":
    main()

pseudo_gt_tcn:   0%|          | 0/16 [00:00<?, ?it/s]


     tile native_shape  n_sources  has_radd  has_gs2  has_gl  pos_frac  neg_frac  ign_frac  pos_mean_conf  strong_pos  medium_pos  pos_y2020  pos_y2021  pos_y2022  pos_y2023  pos_y2024  pos_y2025
18NWG_6_6    1002x1002          3      True     True    True  0.349945  0.430020  0.220035       0.917765      282308       69038      73705      88446      75743      34355      53447      25650
18NWH_1_4    1002x1002          3      True     True    True  0.025219  0.884200  0.090581       0.876045       16612        8708       8282       6455       3994       1475       3940       1174
18NWJ_8_9    1002x1002          3      True     True    True  0.056743  0.802113  0.141144       0.902255       43254       13716      15412      13395      11469       3082       6457       7155
18NWM_9_4      362x362          3      True     True    True  0.038514  0.583903  0.377583       0.937933        4369         678        814       1257        722        859        789        606
18NXH_6_8    1002x1

In [7]:
# pseudo_gt_tcn.py

from pathlib import Path
from datetime import date
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PROCESSED = ROOT / "processed"
CLOUD_ROOT = PROCESSED / "s2_cloudmask"

OUT = Path("eda_artifacts/pseudo_gt_tcn")
OUT.mkdir(parents=True, exist_ok=True)

TARGET_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)
DATE_AGREEMENT_DAYS = 90

WINDOW_START = date(TARGET_YEARS[0], 1, 1).toordinal()
WINDOW_END   = date(TARGET_YEARS[-1] + 1, 1, 1).toordinal()
EPOCH_RADD   = date(2014, 12, 31).toordinal()
EPOCH_GLADS2 = date(2019, 1, 1).toordinal()

# optical labels get downweighted when cloudy
OPTICAL_CLOUD_ATTENUATION = 0.50


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile):
    """Reference grid for the tile. Tries S2, GLAD-L, then RADD."""
    s2_dir = ROOT / f"sentinel-2/train/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best

    gl = sorted((ROOT / "labels/train/gladl").glob(f"gladl_{tile}_alert*.tif"))
    if gl:
        with rasterio.open(gl[0]) as r:
            return r.transform, r.crs, r.shape

    p = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if p.exists():
        with rasterio.open(p) as r:
            return r.transform, r.crs, r.shape

    return None


def reproj(path, ref, dtype):
    out = np.zeros(ref[2], dtype=dtype)
    with rasterio.open(path) as src:
        reproject(
            rasterio.band(src, 1), out,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref[0], dst_crs=ref[1],
            resampling=Resampling.nearest
        )
    return out


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape
    h_tgt, w_tgt = target_shape
    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    return arr[row_idx[:, None], col_idx[None, :]]


def ordinal_to_year_month(ord_arr):
    year = np.zeros(ord_arr.shape, dtype=np.int16)
    month = np.zeros(ord_arr.shape, dtype=np.int8)
    uniq = np.unique(ord_arr[ord_arr > 0])
    for o in uniq:
        d = date.fromordinal(int(o))
        m = ord_arr == o
        year[m] = d.year
        month[m] = d.month
    return year, month


# ============================================================
# CLOUD ACCESS
# ============================================================
def find_cloud_mask_path(tile, year, month):
    tile_dir = CLOUD_ROOT / "train" / f"{tile}__s2_l2a"
    if not tile_dir.exists():
        return None

    candidates = [
        tile_dir / f"{tile}__s2_l2a_{year}_{month}_cloudmask.tif",
        tile_dir / f"{tile}__s2_l2a_{year}_{month:02d}_cloudmask.tif",
    ]
    for c in candidates:
        if c.exists():
            return c

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month:02d}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    return None


def load_cloud_mask_on_ref(tile, year, month, ref):
    """
    Returns cloudy bool raster on ref grid, or None if mask not found.
    Assumes:
      1 = cloud
      0 = clear
    """
    p = find_cloud_mask_path(tile, year, month)
    if p is None:
        return None
    arr = reproj(p, ref, dtype=np.uint8)
    return arr == 1


def cloud_penalty_from_dates(tile, date_ord, ref):
    """
    Returns per-pixel multiplier in [OPTICAL_CLOUD_ATTENUATION, 1.0]
    based on cloud mask at the event month for that source.
    """
    penalty = np.ones(ref[2], dtype=np.float32)
    year_arr, month_arr = ordinal_to_year_month(date_ord)

    pairs = np.unique(np.stack([year_arr, month_arr], axis=-1).reshape(-1, 2), axis=0)
    for y, m in pairs:
        y = int(y)
        m = int(m)
        if y <= 0 or m <= 0:
            continue
        cloudy = load_cloud_mask_on_ref(tile, y, m, ref)
        if cloudy is None:
            continue
        sel = (year_arr == y) & (month_arr == m) & cloudy
        penalty[sel] *= OPTICAL_CLOUD_ATTENUATION

    return penalty


# ============================================================
# LABEL LOADERS
# ============================================================
def get_radd(tile, ref):
    p = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if not p.exists():
        return None, None, False

    raw = reproj(p, ref, dtype=np.int32)
    days = raw % 10000
    date_ord = np.where(raw > 0, days + EPOCH_RADD, 0)
    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (raw > 0) & in_window
    return mask, np.where(mask, date_ord, 0), True


def get_glads2(tile, ref):
    ap = ROOT / f"labels/train/glads2/glads2_{tile}_alert.tif"
    dp = ROOT / f"labels/train/glads2/glads2_{tile}_alertDate.tif"
    if not (ap.exists() and dp.exists()):
        return None, None, False

    alert = reproj(ap, ref, dtype=np.uint8)
    days  = reproj(dp, ref, dtype=np.uint16)
    date_ord = np.where(alert > 0, days.astype(np.int64) + EPOCH_GLADS2, 0)
    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)

    # same original logic: alert >= 2 is a vote
    mask = (alert >= 2) & in_window
    return mask, np.where(mask, date_ord, 0), True


def get_gladl(tile, ref):
    any_file = False
    mask = np.zeros(ref[2], dtype=bool)
    date_ord = np.zeros(ref[2], dtype=np.int64)

    for y in TARGET_YEARS:
        ap = ROOT / f"labels/train/gladl/gladl_{tile}_alert{y%100:02d}.tif"
        dp = ROOT / f"labels/train/gladl/gladl_{tile}_alertDate{y%100:02d}.tif"
        if not (ap.exists() and dp.exists()):
            continue

        any_file = True
        alert = reproj(ap, ref, dtype=np.uint8)
        doy = reproj(dp, ref, dtype=np.uint16)
        year_start = date(y, 1, 1).toordinal()
        year_date = np.where(alert > 0, doy.astype(np.int64) + year_start - 1, 0)
        year_mask = alert > 0

        better = year_mask & ((date_ord == 0) | ((year_date > 0) & (year_date < date_ord)))
        date_ord = np.where(better, year_date, date_ord)
        mask = mask | year_mask

    if not any_file:
        return None, None, False

    return mask, date_ord, True


# ============================================================
# BUILDER
# ============================================================
def build_pgt(tile):
    ref = get_ref(tile)
    if ref is None:
        return None

    native_shape = ref[2]

    radd_mask, radd_date, has_radd = get_radd(tile, ref)
    gs2_mask,  gs2_date,  has_gs2  = get_glads2(tile, ref)
    gl_mask,   gl_date,   has_gl   = get_gladl(tile, ref)

    if radd_mask is None:
        radd_mask = np.zeros(native_shape, dtype=bool)
        radd_date = np.zeros(native_shape, dtype=np.int64)

    if gs2_mask is None:
        gs2_mask = np.zeros(native_shape, dtype=bool)
        gs2_date = np.zeros(native_shape, dtype=np.int64)

    if gl_mask is None:
        gl_mask = np.zeros(native_shape, dtype=bool)
        gl_date = np.zeros(native_shape, dtype=np.int64)

    votes = radd_mask.astype(np.int8) + gs2_mask.astype(np.int8) + gl_mask.astype(np.int8)
    n_sources = sum([has_radd, has_gs2, has_gl])

    # ORIGINAL confidence structure
    confidence = votes.astype(np.float32) * 0.4

    def pair_bonus(m1, d1, m2, d2):
        both = m1 & m2
        if not both.any():
            return np.zeros(native_shape, dtype=np.float32)
        diff = np.abs(d1.astype(np.int64) - d2.astype(np.int64))
        close = both & (diff <= DATE_AGREEMENT_DAYS) & (d1 > 0) & (d2 > 0)
        return np.where(close, 0.15, 0.0).astype(np.float32)

    confidence += pair_bonus(radd_mask, radd_date, gs2_mask, gs2_date)
    confidence += pair_bonus(radd_mask, radd_date, gl_mask,  gl_date)
    confidence += pair_bonus(gs2_mask,  gs2_date,  gl_mask,  gl_date)

    # ONLY NEW THING:
    # downweight optical-derived confidence where cloudy at source event month
    if has_gs2:
        gs2_cloud_penalty = cloud_penalty_from_dates(tile, gs2_date, ref)
        confidence = np.where(gs2_mask, confidence * gs2_cloud_penalty, confidence)

    if has_gl:
        gl_cloud_penalty = cloud_penalty_from_dates(tile, gl_date, ref)
        confidence = np.where(gl_mask, confidence * gl_cloud_penalty, confidence)

    confidence = np.clip(confidence, 0.0, 1.0)

    # ORIGINAL label logic
    label = np.full(native_shape, np.nan, dtype=np.float32)
    if n_sources >= 2:
        label[votes >= 2] = 1.0
        label[votes == 0] = 0.0
    elif n_sources == 1:
        label[votes == 0] = 0.0

    confidence = np.where(np.isnan(label), 0.0, confidence)
    confidence = np.where(label == 0.0, 1.0, confidence)

    # ORIGINAL event date logic: earliest across sources
    event_month = np.zeros(native_shape, dtype=np.int8)
    event_year = np.zeros(native_shape, dtype=np.int16)

    dates_stack = np.stack([radd_date, gs2_date, gl_date])
    masked = np.where(dates_stack > 0, dates_stack, np.iinfo(np.int64).max)
    earliest = masked.min(axis=0)
    has_date = earliest < np.iinfo(np.int64).max

    if has_date.any():
        for ord_val in np.unique(earliest[has_date]):
            m = (earliest == ord_val) & has_date & (label == 1.0)
            if m.any():
                d = date.fromordinal(int(ord_val))
                event_month[m] = d.month
                event_year[m] = d.year

    # ORIGINAL resize to 1000x1000
    label_r       = resize_nearest(label, TARGET_SHAPE)
    confidence_r  = resize_nearest(confidence, TARGET_SHAPE)
    event_month_r = resize_nearest(event_month, TARGET_SHAPE)
    event_year_r  = resize_nearest(event_year, TARGET_SHAPE)

    pos = label_r == 1.0
    total = label_r.size

    per_year = {}
    for y in TARGET_YEARS:
        per_year[f"pos_y{y}"] = int(((event_year_r == y) & pos).sum())

    return {
        "label": label_r,
        "confidence": confidence_r,
        "event_month": event_month_r,
        "event_year": event_year_r,
        "native_shape": f"{native_shape[0]}x{native_shape[1]}",
        "n_sources": n_sources,
        "has": {"radd": has_radd, "gs2": has_gs2, "gl": has_gl},
        "pos_frac": float(pos.sum() / total),
        "neg_frac": float((label_r == 0.0).sum() / total),
        "ign_frac": float(np.isnan(label_r).sum() / total),
        "pos_mean_conf": float(confidence_r[pos].mean()) if pos.any() else 0.0,
        "strong_pos": int((pos & (confidence_r >= 0.95)).sum()),
        "medium_pos": int((pos & (confidence_r >= 0.80) & (confidence_r < 0.95)).sum()),
        "per_year": per_year,
    }


# ============================================================
# RUN
# ============================================================
train_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/train").iterdir()
])

stats = []
for tile in tqdm(train_tiles, desc=f"pseudo-GT @ {TARGET_SHAPE}"):
    try:
        r = build_pgt(tile)
        if r is None:
            print(f"  {tile}: skipped")
            continue

        np.savez_compressed(
            OUT / f"{tile}.npz",
            label=r["label"],
            confidence=r["confidence"],
            event_month=r["event_month"],
            event_year=r["event_year"],
        )

        row = {
            "tile": tile,
            "native_shape": r["native_shape"],
            "n_sources": r["n_sources"],
            "has_radd": r["has"]["radd"],
            "has_gs2": r["has"]["gs2"],
            "has_gl": r["has"]["gl"],
            "pos_frac": r["pos_frac"],
            "neg_frac": r["neg_frac"],
            "ign_frac": r["ign_frac"],
            "pos_mean_conf": r["pos_mean_conf"],
            "strong_pos": r["strong_pos"],
            "medium_pos": r["medium_pos"],
        }
        row.update(r["per_year"])
        stats.append(row)

    except Exception as e:
        print(f"  {tile}: ERROR {e}")

df = pd.DataFrame(stats)
df.to_csv(OUT / "stats.csv", index=False)

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 30)
print("\n" + df.to_string(index=False))
print(f"\nTotal tiles: {len(df)}")
print(f"Mean pos_frac: {df.pos_frac.mean():.3%}")
print(f"Total strong pos: {df.strong_pos.sum():,}")
print(f"Total medium pos: {df.medium_pos.sum():,}")
print(f"\nPositives per year:")
for y in TARGET_YEARS:
    print(f"  {y}: {df[f'pos_y{y}'].sum():,}")

pseudo-GT @ (1000, 1000):   0%|          | 0/16 [00:00<?, ?it/s]


     tile native_shape  n_sources  has_radd  has_gs2  has_gl  pos_frac  neg_frac  ign_frac  pos_mean_conf  strong_pos  medium_pos  pos_y2020  pos_y2021  pos_y2022  pos_y2023  pos_y2024  pos_y2025
18NWG_6_6    1002x1002          3      True     True    True  0.365629  0.550460  0.083911       0.768296      173128       72575     108504     107603      57758      33338      37755      20671
18NWH_1_4    1002x1002          3      True     True    True  0.025868  0.950983  0.023149       0.775986       10725        7726      13654       5199       2859        823       2777        556
18NWJ_8_9    1002x1002          3      True     True    True  0.058639  0.892211  0.049150       0.828775       31382       17639      26172      11559      10718        483       4905       4802
18NWM_9_4      362x362          3      True     True    True  0.038479  0.913840  0.047681       0.927377       30026        8453       9048       9846       6144       5441       4224       3776
18NXH_6_8    1002x1

In [19]:
#Codex pseusdo GT 

# pseudo_gt.py

from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from tqdm.auto import tqdm


ROOT = Path("data/makeathon-challenge")
CLOUD_ROOT = ROOT / "processed/s2_cloudmask"
OUT = Path("eda_artifacts/pseudo_gt_codex")
OUT.mkdir(parents=True, exist_ok=True)

TARGET_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)

DATE_AGREEMENT_DAYS = 90

WINDOW_START = date(2020, 1, 1).toordinal()
WINDOW_END = date(2026, 1, 1).toordinal()

EPOCH_RADD = date(2014, 12, 31).toordinal()
EPOCH_GLADS2 = date(2019, 1, 1).toordinal()

RADD_WEIGHT_CLEAR = 0.55
RADD_WEIGHT_CLOUDY = 0.80

OPTICAL_WEIGHT_CLEAR = 0.35
OPTICAL_WEIGHT_CLOUDY = 0.10

RADAR_OPTICAL_PAIR_BONUS = 0.10
OPTICAL_PAIR_BONUS_CLEAR = 0.15
OPTICAL_PAIR_BONUS_CLOUDY = 0.02

POSITIVE_CONF_THRESHOLD = 0.50
NEGATIVE_CONFIDENCE = 1.00


def get_ref(tile):
    s2_dir = ROOT / f"sentinel-2/train/{tile}__s2_l2a"
    if s2_dir.exists():
        best = None
        best_area = 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as src:
                h, w = src.shape
                area = h * w
                if h >= 300 and w >= 300 and area > best_area:
                    best_area = area
                    best = (src.transform, src.crs, src.shape)
        if best is not None:
            return best

    gl_files = sorted((ROOT / "labels/train/gladl").glob(f"gladl_{tile}_alert*.tif"))
    if gl_files:
        with rasterio.open(gl_files[0]) as src:
            return src.transform, src.crs, src.shape

    radd_path = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if radd_path.exists():
        with rasterio.open(radd_path) as src:
            return src.transform, src.crs, src.shape

    return None


def reproj(path, ref, dtype):
    out = np.zeros(ref[2], dtype=dtype)
    with rasterio.open(path) as src:
        reproject(
            rasterio.band(src, 1),
            out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref[0],
            dst_crs=ref[1],
            resampling=Resampling.nearest,
        )
    return out


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape
    h_tgt, w_tgt = target_shape

    rows = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    cols = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)

    return arr[rows[:, None], cols[None, :]]


def ordinal_to_year_month(ord_arr):
    year = np.zeros(ord_arr.shape, dtype=np.int16)
    month = np.zeros(ord_arr.shape, dtype=np.int8)

    for ord_val in np.unique(ord_arr[ord_arr > 0]):
        d = date.fromordinal(int(ord_val))
        sel = ord_arr == ord_val
        year[sel] = d.year
        month[sel] = d.month

    return year, month


def find_cloud_mask_path(tile, year, month):
    tile_dir = CLOUD_ROOT / "train" / f"{tile}__s2_l2a"
    if not tile_dir.exists():
        return None

    candidates = [
        tile_dir / f"{tile}__s2_l2a_{year}_{month}_cloudmask.tif",
        tile_dir / f"{tile}__s2_l2a_{year}_{month:02d}_cloudmask.tif",
    ]

    for p in candidates:
        if p.exists():
            return p

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month:02d}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    return None


def load_cloud_mask_on_ref(tile, year, month, ref):
    p = find_cloud_mask_path(tile, year, month)
    if p is None:
        return None

    arr = reproj(p, ref, dtype=np.uint8)
    return arr == 1


def cloudy_from_dates(tile, date_ord, ref):
    cloudy_any = np.zeros(ref[2], dtype=bool)

    year_arr, month_arr = ordinal_to_year_month(date_ord)
    pairs = np.unique(
        np.stack([year_arr, month_arr], axis=-1).reshape(-1, 2),
        axis=0,
    )

    for y, m in pairs:
        y = int(y)
        m = int(m)

        if y <= 0 or m <= 0:
            continue

        cloudy = load_cloud_mask_on_ref(tile, y, m, ref)
        if cloudy is None:
            continue

        sel = (year_arr == y) & (month_arr == m)
        cloudy_any[sel] = cloudy[sel]

    return cloudy_any


def get_radd(tile, ref):
    p = ROOT / f"labels/train/radd/radd_{tile}_labels.tif"
    if not p.exists():
        return None, None, False

    raw = reproj(p, ref, dtype=np.int32)

    days = raw % 10000
    date_ord = np.where(raw > 0, days + EPOCH_RADD, 0)

    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (raw > 0) & in_window

    return mask, np.where(mask, date_ord, 0), True


def get_glads2(tile, ref):
    alert_path = ROOT / f"labels/train/glads2/glads2_{tile}_alert.tif"
    date_path = ROOT / f"labels/train/glads2/glads2_{tile}_alertDate.tif"

    if not alert_path.exists() or not date_path.exists():
        return None, None, False

    alert = reproj(alert_path, ref, dtype=np.uint8)
    days = reproj(date_path, ref, dtype=np.uint16)

    date_ord = np.where(alert > 0, days.astype(np.int64) + EPOCH_GLADS2, 0)

    in_window = (date_ord >= WINDOW_START) & (date_ord < WINDOW_END)
    mask = (alert >= 2) & in_window

    return mask, np.where(mask, date_ord, 0), True


def get_gladl(tile, ref):
    any_file = False
    mask = np.zeros(ref[2], dtype=bool)
    date_ord = np.zeros(ref[2], dtype=np.int64)

    for year in TARGET_YEARS:
        yy = year % 100

        alert_path = ROOT / f"labels/train/gladl/gladl_{tile}_alert{yy:02d}.tif"
        date_path = ROOT / f"labels/train/gladl/gladl_{tile}_alertDate{yy:02d}.tif"

        if not alert_path.exists() or not date_path.exists():
            continue

        any_file = True

        alert = reproj(alert_path, ref, dtype=np.uint8)
        doy = reproj(date_path, ref, dtype=np.uint16)

        year_start = date(year, 1, 1).toordinal()
        this_date = np.where(alert > 0, doy.astype(np.int64) + year_start - 1, 0)
        this_mask = alert > 0

        better = this_mask & (
            (date_ord == 0) | ((this_date > 0) & (this_date < date_ord))
        )

        date_ord = np.where(better, this_date, date_ord)
        mask |= this_mask

    if not any_file:
        return None, None, False

    return mask, date_ord, True


def close_date_mask(m1, d1, m2, d2):
    both = m1 & m2
    if not both.any():
        return np.zeros(m1.shape, dtype=bool)

    diff = np.abs(d1.astype(np.int64) - d2.astype(np.int64))
    return both & (diff <= DATE_AGREEMENT_DAYS) & (d1 > 0) & (d2 > 0)


def pair_bonus(m1, d1, m2, d2, value):
    close = close_date_mask(m1, d1, m2, d2)
    return np.where(close, value, 0.0).astype(np.float32)


def optical_pair_bonus(tile, gs2_mask, gs2_date, gl_mask, gl_date, ref):
    close = close_date_mask(gs2_mask, gs2_date, gl_mask, gl_date)
    if not close.any():
        return np.zeros(ref[2], dtype=np.float32)

    gs2_cloudy = cloudy_from_dates(tile, gs2_date, ref)
    gl_cloudy = cloudy_from_dates(tile, gl_date, ref)

    cloudy_pair = gs2_cloudy | gl_cloudy

    bonus = np.where(
        cloudy_pair,
        OPTICAL_PAIR_BONUS_CLOUDY,
        OPTICAL_PAIR_BONUS_CLEAR,
    )

    return np.where(close, bonus, 0.0).astype(np.float32)


def build_confidence(tile, ref, radd_mask, radd_date, gs2_mask, gs2_date, gl_mask, gl_date):
    radd_cloudy = cloudy_from_dates(tile, radd_date, ref)
    gs2_cloudy = cloudy_from_dates(tile, gs2_date, ref)
    gl_cloudy = cloudy_from_dates(tile, gl_date, ref)

    radd_weight = np.where(
        radd_cloudy,
        RADD_WEIGHT_CLOUDY,
        RADD_WEIGHT_CLEAR,
    ).astype(np.float32)

    gs2_weight = np.where(
        gs2_cloudy,
        OPTICAL_WEIGHT_CLOUDY,
        OPTICAL_WEIGHT_CLEAR,
    ).astype(np.float32)

    gl_weight = np.where(
        gl_cloudy,
        OPTICAL_WEIGHT_CLOUDY,
        OPTICAL_WEIGHT_CLEAR,
    ).astype(np.float32)

    confidence = np.zeros(ref[2], dtype=np.float32)

    confidence += np.where(radd_mask, radd_weight, 0.0).astype(np.float32)
    confidence += np.where(gs2_mask, gs2_weight, 0.0).astype(np.float32)
    confidence += np.where(gl_mask, gl_weight, 0.0).astype(np.float32)

    confidence += pair_bonus(
        radd_mask,
        radd_date,
        gs2_mask,
        gs2_date,
        RADAR_OPTICAL_PAIR_BONUS,
    )

    confidence += pair_bonus(
        radd_mask,
        radd_date,
        gl_mask,
        gl_date,
        RADAR_OPTICAL_PAIR_BONUS,
    )

    confidence += optical_pair_bonus(
        tile,
        gs2_mask,
        gs2_date,
        gl_mask,
        gl_date,
        ref,
    )

    return np.clip(confidence, 0.0, 1.0)


def build_pseudo_gt(tile):
    ref = get_ref(tile)
    if ref is None:
        return None

    shape = ref[2]

    radd_mask, radd_date, has_radd = get_radd(tile, ref)
    gs2_mask, gs2_date, has_gs2 = get_glads2(tile, ref)
    gl_mask, gl_date, has_gl = get_gladl(tile, ref)

    if radd_mask is None:
        radd_mask = np.zeros(shape, dtype=bool)
        radd_date = np.zeros(shape, dtype=np.int64)

    if gs2_mask is None:
        gs2_mask = np.zeros(shape, dtype=bool)
        gs2_date = np.zeros(shape, dtype=np.int64)

    if gl_mask is None:
        gl_mask = np.zeros(shape, dtype=bool)
        gl_date = np.zeros(shape, dtype=np.int64)

    votes = (
        radd_mask.astype(np.int8)
        + gs2_mask.astype(np.int8)
        + gl_mask.astype(np.int8)
    )

    n_sources = int(has_radd) + int(has_gs2) + int(has_gl)

    confidence = build_confidence(
        tile,
        ref,
        radd_mask,
        radd_date,
        gs2_mask,
        gs2_date,
        gl_mask,
        gl_date,
    )

    label = np.full(shape, np.nan, dtype=np.float32)

    label[confidence >= POSITIVE_CONF_THRESHOLD] = 1.0

    if n_sources > 0:
        label[votes == 0] = 0.0

    label[(votes > 0) & (confidence < POSITIVE_CONF_THRESHOLD)] = np.nan

    confidence = np.where(np.isnan(label), 0.0, confidence)
    confidence = np.where(label == 0.0, NEGATIVE_CONFIDENCE, confidence)

    event_month = np.zeros(shape, dtype=np.int8)
    event_year = np.zeros(shape, dtype=np.int16)

    dates = np.stack([radd_date, gs2_date, gl_date])
    masked_dates = np.where(dates > 0, dates, np.iinfo(np.int64).max)

    earliest = masked_dates.min(axis=0)
    has_date = earliest < np.iinfo(np.int64).max

    for ord_val in np.unique(earliest[has_date]):
        sel = (earliest == ord_val) & has_date & (label == 1.0)
        if not sel.any():
            continue

        d = date.fromordinal(int(ord_val))
        event_month[sel] = d.month
        event_year[sel] = d.year

    label = resize_nearest(label, TARGET_SHAPE)
    confidence = resize_nearest(confidence, TARGET_SHAPE)
    event_month = resize_nearest(event_month, TARGET_SHAPE)
    event_year = resize_nearest(event_year, TARGET_SHAPE)

    return {
        "label": label,
        "confidence": confidence,
        "event_month": event_month,
        "event_year": event_year,
        "native_shape": f"{shape[0]}x{shape[1]}",
        "n_sources": n_sources,
        "has_radd": has_radd,
        "has_gs2": has_gs2,
        "has_gl": has_gl,
    }


train_root = ROOT / "sentinel-2/train"

train_tiles = sorted(
    p.name.replace("__s2_l2a", "")
    for p in train_root.iterdir()
    if p.is_dir() and p.name.endswith("__s2_l2a")
)

stats = []

for tile in tqdm(train_tiles, desc="pseudo GT"):
    try:
        result = build_pseudo_gt(tile)
        if result is None:
            print(f"{tile}: skipped")
            continue

        np.savez_compressed(
            OUT / f"{tile}.npz",
            label=result["label"],
            confidence=result["confidence"],
            event_month=result["event_month"],
            event_year=result["event_year"],
        )

        label = result["label"]
        confidence = result["confidence"]
        event_year = result["event_year"]

        pos = label == 1.0
        neg = label == 0.0
        ign = np.isnan(label)

        row = {
            "tile": tile,
            "native_shape": result["native_shape"],
            "n_sources": result["n_sources"],
            "has_radd": result["has_radd"],
            "has_gs2": result["has_gs2"],
            "has_gl": result["has_gl"],
            "pos_frac": float(pos.mean()),
            "neg_frac": float(neg.mean()),
            "ign_frac": float(ign.mean()),
            "pos_mean_conf": float(confidence[pos].mean()) if pos.any() else 0.0,
            "strong_pos": int((pos & (confidence >= 0.95)).sum()),
            "medium_pos": int((pos & (confidence >= 0.80) & (confidence < 0.95)).sum()),
            "weak_pos": int((pos & (confidence < 0.80)).sum()),
        }

        for year in TARGET_YEARS:
            row[f"pos_y{year}"] = int((pos & (event_year == year)).sum())

        stats.append(row)

    except Exception as e:
        print(f"{tile}: ERROR {e}")


df = pd.DataFrame(stats)
df.to_csv(OUT / "stats.csv", index=False)

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 50)

print(df.to_string(index=False))
print(f"\nTotal tiles: {len(df)}")

if len(df) > 0:
    print(f"Mean pos_frac: {df.pos_frac.mean():.3%}")
    print(f"Mean neg_frac: {df.neg_frac.mean():.3%}")
    print(f"Mean ign_frac: {df.ign_frac.mean():.3%}")
    print(f"Total strong pos: {df.strong_pos.sum():,}")
    print(f"Total medium pos: {df.medium_pos.sum():,}")
    print(f"Total weak pos: {df.weak_pos.sum():,}")

    print("\nPositives per year:")
    for year in TARGET_YEARS:
        print(f"  {year}: {df[f'pos_y{year}'].sum():,}")


pseudo GT:   0%|          | 0/16 [00:00<?, ?it/s]

     tile native_shape  n_sources  has_radd  has_gs2  has_gl  pos_frac  neg_frac  ign_frac  pos_mean_conf  strong_pos  medium_pos  weak_pos  pos_y2020  pos_y2021  pos_y2022  pos_y2023  pos_y2024  pos_y2025
18NWG_6_6    1002x1002          3      True     True    True  0.362415  0.550460  0.087125       0.946522      267364       59646     35405     114284     106803      55769      33365      36523      15671
18NWH_1_4    1002x1002          3      True     True    True  0.034329  0.950983  0.014688       0.864596       17441        8398      8490      16409       6849       3841       1792       4412       1026
18NWJ_8_9    1002x1002          3      True     True    True  0.070155  0.892211  0.037634       0.883339       35876       20589     13690      34928      12912      11204       1306       5145       4660
18NWM_9_4      362x362          3      True     True    True  0.058609  0.913840  0.027551       0.810060       23050       14519     21040      13535      12788       9554    

In [9]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import lightgbm as lgb
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import sys, json

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("/shared-docker/data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt_tcn")
MODEL_DIR = ROOT / "processed" / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR = ROOT / "processed" / "submission"
SUB_DIR.mkdir(parents=True, exist_ok=True)
TILE_PRED_DIR = ROOT / "processed" / "predictions"
TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)   # 384 + 5 = 389

NEG_RATIO = 5
N_BOOST_ROUND = 400
CV_HOLDOUT = "18NXH_6_8"   # held-out validation tile


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    """Reference grid for tile (tries S2, then AEF)."""
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best
    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape
    return None


def load_aef_to_ref(tile, year, split, ref):
    """Load AEF, reproject all 64 bands to reference grid."""
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"
    if not p.exists():
        return None

    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)
        for b in range(N_AEF_BANDS):
            reproject(
                raw[b], aef[b],
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=ref[0], dst_crs=ref[1],
                resampling=Resampling.bilinear
            )
    return aef


def feature_names():
    names = []
    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")
    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i+1]}")
    return names


def extract_features(tile, split):
    """
    Returns features (N_pixels, N_FEATURES) and reference grid for tile.
    Features: 6×64 AEF values + 5 year-over-year cosine similarities.
    IMPORTANT: now uses native tile shape, no forced resize to 1000x1000.
    """
    ref = get_ref(tile, split)
    if ref is None:
        return None, None

    native_shape = ref[2]

    aef_stack = []
    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)
        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None
        if aef.shape[1:] != native_shape:
            raise ValueError(f"AEF shape mismatch for {tile} {year}: {aef.shape[1:]} vs {native_shape}")
        aef_stack.append(aef)

    changes = []
    for i in range(len(YEARS) - 1):
        a, b = aef_stack[i], aef_stack[i + 1]
        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b
        cos_sim = np.where(denom > 1e-6, dot / (denom + 1e-9), 0.0).astype(np.float32)
        changes.append(cos_sim)

    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0
    )
    features = all_feats.reshape(N_FEATURES, -1).T
    return features, ref


# ============================================================
# STEP 1 — Extract features for training
# ============================================================
print(f"=== Step 1: Extract features (target = {N_FEATURES} per pixel) ===")
train_tiles = sorted([p.name.replace("__s2_l2a", "")
                      for p in (ROOT / "sentinel-2/train").iterdir()])

all_feats, all_labels, all_weights, all_tiles = [], [], [], []

for tile in tqdm(train_tiles, desc="train features"):
    pgt_path = PGT_DIR / f"{tile}.npz"
    if not pgt_path.exists():
        continue

    feats, ref = extract_features(tile, "train")
    if feats is None:
        continue

    pgt = np.load(pgt_path)
    label = pgt["label"].reshape(-1)

    # NEW: use training weight from pseudo_gt_tcn
    if "weight" in pgt.files:
        weight = pgt["weight"].reshape(-1)
    elif "weight_old" in pgt.files:
        weight = pgt["weight_old"].reshape(-1)
    else:
        weight = pgt["confidence"].reshape(-1)

    # NEW: ignore label is -1, not NaN
    valid = label != -1

    all_feats.append(feats[valid])
    all_labels.append(label[valid])
    all_weights.append(weight[valid])
    all_tiles.append(np.full(int(valid.sum()), tile))

X = np.concatenate(all_feats, axis=0)
y = np.concatenate(all_labels).astype(np.float32)
w = np.concatenate(all_weights).astype(np.float32)
tile_ids = np.concatenate(all_tiles)

print(f"\nTotal labeled pixels: {len(y):,}")
print(f"Positives: {int((y == 1).sum()):,} ({(y == 1).mean():.2%})")
print(f"Negatives: {int((y == 0).sum()):,}")
print(f"Feature matrix: {X.shape}, dtype={X.dtype}")
print(f"Memory: {X.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 2 — Subsample negatives
# ============================================================
print(f"\n=== Step 2: Subsample negatives {NEG_RATIO}x ===")
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_RATIO)
rng = np.random.default_rng(42)
neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)

keep = np.concatenate([pos_idx, neg_sampled])
rng.shuffle(keep)

X_kept = X[keep]
y_kept = y[keep]
w_kept = w[keep]
tiles_kept = tile_ids[keep]

print(f"After subsampling: {len(y_kept):,} rows ({(y_kept == 1).mean():.1%} positive)")
print(f"Memory: {X_kept.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 3 — Held-out validation for threshold
# ============================================================
val_mask = tiles_kept == CV_HOLDOUT
tr_mask = ~val_mask

X_tr = X_kept[tr_mask]
y_tr = y_kept[tr_mask]
w_tr = w_kept[tr_mask]

X_val = X_kept[val_mask]
y_val = y_kept[val_mask]

print(f"\nValidation tile: {CV_HOLDOUT}")
print(f"  Train rows: {len(y_tr):,}, Val rows: {len(y_val):,}")


# ============================================================
# STEP 4 — Train LightGBM
# ============================================================
print("\n=== Step 4: Train LightGBM ===")
names = feature_names()

lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr, feature_name=names)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=N_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

model.save_model(str(MODEL_DIR / "lgbm_allyears.txt"))


# ============================================================
# STEP 5 — Tune threshold
# ============================================================
print("\n=== Step 5: Tune threshold ===")
val_pred = model.predict(X_val)

best_thr, best_f1 = 0.5, 0
for thr in np.linspace(0.1, 0.9, 17):
    f1 = f1_score(y_val, val_pred > thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

print(f"Best threshold: {best_thr:.2f} (F1={best_f1:.3f})")


# ============================================================
# STEP 6 — Predict test tiles
# ============================================================
print("\n=== Step 6: Predict test tiles ===")
test_tiles = sorted([p.name.replace("__s2_l2a", "")
                     for p in (ROOT / "sentinel-2/test").iterdir()])
print(f"Test tiles: {test_tiles}")

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue

    pred = model.predict(feats)

    # NEW: keep native shape
    h_native, w_native = ref[2]
    binary_native = (pred > best_thr).astype(np.uint8).reshape(h_native, w_native)

    tile_path = TILE_PRED_DIR / f"{tile}_pred.tif"
    profile = {
        "driver": "GTiff",
        "height": h_native,
        "width": w_native,
        "count": 1,
        "dtype": "uint8",
        "crs": ref[1],
        "transform": ref[0],
        "nodata": 0,
    }

    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)

    geojson = raster_to_geojson(str(tile_path), output_path=None)
    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile

    all_features_geojson.extend(geojson["features"])
    print(f"  {tile}: {int(binary_native.sum()):,} positive pixels, {len(geojson['features'])} polygons")


# ============================================================
# STEP 7 — Save submission
# ============================================================
submission = {"type": "FeatureCollection", "features": all_features_geojson}

sub_path = SUB_DIR / "submission.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\n=== Submission saved: {sub_path} ===")
print(f"Total polygons: {len(all_features_geojson)}")


# ============================================================
# STEP 8 — Feature importance
# ============================================================
print("\n=== Feature importance (top 15) ===")
importance = pd.DataFrame({
    "feature": names,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
print(importance.head(15).to_string(index=False))

print("\n=== Importance by year ===")
by_year = {y: 0 for y in YEARS}
change_total = 0

for _, row in importance.iterrows():
    feat = row["feature"]
    if feat.startswith("aef_"):
        year = int(feat.split("_")[1])
        by_year[year] += row["gain"]
    elif feat.startswith("change_"):
        change_total += row["gain"]

for y in YEARS:
    print(f"  AEF {y}: {by_year[y]:.0f}")
print(f"  All changes: {change_total:.0f}")

=== Step 1: Extract features (target = 389 per pixel) ===


train features:   0%|          | 0/16 [00:00<?, ?it/s]

IndexError: boolean index did not match indexed array along dimension 0; dimension is 1004004 but corresponding boolean dimension is 1000000

In [ ]:
from pathlib import Path
from datetime import date
import json

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS
from rasterio.transform import Affine
from sklearn.decomposition import IncrementalPCA
import lightgbm as lgb
from tqdm.auto import tqdm

from submission_utils import raster_to_geojson


# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt")

OUT_DIR = Path("eda_artifacts/quick_lgbm")
MODEL_DIR = OUT_DIR / "models"
PRED_DIR = OUT_DIR / "predictions"
SUB_DIR = Path("submission")

for d in [OUT_DIR, MODEL_DIR, PRED_DIR, SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)

N_AEF_BANDS = 64
N_AEF_PCA = 8

AEF_YEAR = 2020

PCA_SAMPLE_PER_TILE = 40000
TRAIN_POS_PER_TILE = 120000
TRAIN_NEG_PER_TILE = 240000
RANDOM_SEED = 42

THRESHOLD = 0.17

CLOUD_ROOT = ROOT / "processed/s2_cloudmask"


# ============================================================
# BASIC GRID HELPERS
# ============================================================
def get_ref(tile, split):
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if s2_dir.exists():
        best = None
        best_area = 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as src:
                h, w = src.shape
                area = h * w
                if h >= 300 and w >= 300 and area > best_area:
                    best_area = area
                    best = (src.transform, src.crs, src.shape)
        if best is not None:
            return best

    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as src:
            return src.transform, src.crs, src.shape

    return None


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape

    rows = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    cols = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)

    if arr.ndim == 2:
        return arr[rows[:, None], cols[None, :]]

    return arr[:, rows[:, None], cols[None, :]]


def resize_bilinear_to_ref(src_arr, src, ref, dtype=np.float32):
    out = np.zeros(ref[2], dtype=dtype)
    reproject(
        src_arr,
        out,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref[0],
        dst_crs=ref[1],
        resampling=Resampling.bilinear,
    )
    return out


def reproject_nearest_band(path, ref, dtype=np.uint8):
    out = np.zeros(ref[2], dtype=dtype)
    with rasterio.open(path) as src:
        reproject(
            rasterio.band(src, 1),
            out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref[0],
            dst_crs=ref[1],
            resampling=Resampling.nearest,
        )
    return out


# ============================================================
# CLOUD MASK
# ============================================================
def find_cloud_mask(tile, split, year, month):
    tile_dir = CLOUD_ROOT / split / f"{tile}__s2_l2a"
    if not tile_dir.exists():
        return None

    candidates = [
        tile_dir / f"{tile}__s2_l2a_{year}_{month}_cloudmask.tif",
        tile_dir / f"{tile}__s2_l2a_{year}_{month:02d}_cloudmask.tif",
    ]

    for p in candidates:
        if p.exists():
            return p

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    fuzzy = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_{month:02d}*_cloudmask.tif"))
    if fuzzy:
        return fuzzy[0]

    return None


def parse_year_month(path):
    parts = path.stem.split("_")
    year = None
    month = None

    for i, part in enumerate(parts):
        if part.isdigit() and len(part) == 4:
            year = int(part)
            if i + 1 < len(parts):
                try:
                    month = int(parts[i + 1])
                except ValueError:
                    month = None
            break

    return year, month


# ============================================================
# AEF PCA
# ============================================================
def load_aef_2020(tile, split, ref):
    p = ROOT / f"aef-embeddings/{split}/{tile}_{AEF_YEAR}.tiff"
    if not p.exists():
        return None

    out = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)

    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)

        for b in range(N_AEF_BANDS):
            reproject(
                raw[b],
                out[b],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref[0],
                dst_crs=ref[1],
                resampling=Resampling.bilinear,
            )

    out = resize_nearest(out, TARGET_SHAPE)
    return out


def fit_aef_pca(train_tiles):
    rng = np.random.default_rng(RANDOM_SEED)
    ipca = IncrementalPCA(n_components=N_AEF_PCA, batch_size=20000)

    print("\n=== Fitting AEF PCA ===")

    for tile in tqdm(train_tiles, desc="AEF PCA"):
        ref = get_ref(tile, "train")
        if ref is None:
            continue

        aef = load_aef_2020(tile, "train", ref)
        if aef is None:
            continue

        flat = aef.reshape(N_AEF_BANDS, -1).T
        valid = np.isfinite(flat).all(axis=1) & (np.abs(flat).sum(axis=1) > 0)

        idx = np.where(valid)[0]
        if len(idx) == 0:
            continue

        n = min(PCA_SAMPLE_PER_TILE, len(idx))
        idx = rng.choice(idx, size=n, replace=False)

        ipca.partial_fit(flat[idx])

    return ipca


def transform_aef_pca(tile, split, ref, pca):
    aef = load_aef_2020(tile, split, ref)
    if aef is None:
        return np.zeros((N_AEF_PCA + 1, *TARGET_SHAPE), dtype=np.float32)

    flat = aef.reshape(N_AEF_BANDS, -1).T
    valid = np.isfinite(flat).all(axis=1) & (np.abs(flat).sum(axis=1) > 0)

    out = np.zeros((flat.shape[0], N_AEF_PCA), dtype=np.float32)
    if valid.any():
        out[valid] = pca.transform(flat[valid]).astype(np.float32)

    out = out.T.reshape(N_AEF_PCA, *TARGET_SHAPE)
    valid_band = valid.reshape(TARGET_SHAPE).astype(np.float32)

    return np.concatenate([out, valid_band[None]], axis=0)


# ============================================================
# S2 FEATURES
# ============================================================
def load_s2_year_features(tile, year, split, ref):
    """
    Fast S2 yearly summaries from cloud-masked monthly observations.

    Returns:
      NDVI mean/min/final/drop
      NDMI mean/min/final/drop
      NBR  mean/min/final/drop
      valid count
    Shape: (13, 1000, 1000)
    """
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if not s2_dir.exists():
        return np.zeros((13, *TARGET_SHAPE), dtype=np.float32)

    files = sorted(s2_dir.glob(f"*_{year}_*.tif"))

    ndvi_list = []
    ndmi_list = []
    nbr_list = []

    for f in files:
        y, m = parse_year_month(f)
        if y != year or m is None:
            continue

        cloud_path = find_cloud_mask(tile, split, y, m)

        with rasterio.open(f) as src:
            if src.shape[0] < 100 or src.shape[1] < 100:
                continue

            try:
                # Sentinel-2 packaged band assumption:
                # B4 red = 4, B8 nir = 8, B11 swir1 = 11, B12 swir2 = 12.
                red, nir, swir1, swir2 = src.read([4, 8, 11, 12]).astype(np.float32) / 10000.0
            except Exception:
                continue

            red_r = resize_bilinear_to_ref(red, src, ref)
            nir_r = resize_bilinear_to_ref(nir, src, ref)
            swir1_r = resize_bilinear_to_ref(swir1, src, ref)
            swir2_r = resize_bilinear_to_ref(swir2, src, ref)

        valid = (red_r > 0) & (nir_r > 0) & (swir1_r > 0) & (swir2_r > 0)

        if cloud_path is not None:
            cloudy = reproject_nearest_band(cloud_path, ref, dtype=np.uint8) == 1
            valid &= ~cloudy

        ndvi = (nir_r - red_r) / (nir_r + red_r + 1e-6)
        ndmi = (nir_r - swir1_r) / (nir_r + swir1_r + 1e-6)
        nbr = (nir_r - swir2_r) / (nir_r + swir2_r + 1e-6)

        ndvi_list.append(np.where(valid, ndvi, np.nan))
        ndmi_list.append(np.where(valid, ndmi, np.nan))
        nbr_list.append(np.where(valid, nbr, np.nan))

    if len(ndvi_list) == 0:
        return np.zeros((13, *TARGET_SHAPE), dtype=np.float32)

    ndvi = np.stack(ndvi_list)
    ndmi = np.stack(ndmi_list)
    nbr = np.stack(nbr_list)

    valid_count = np.isfinite(ndvi).sum(axis=0).astype(np.float32)

    def stats(x):
        with np.errstate(all="ignore"):
            mean = np.nanmean(x, axis=0)
            minv = np.nanmin(x, axis=0)
            final = x[-1]
            drop = np.nanmin(np.diff(x, axis=0), axis=0) if x.shape[0] > 1 else np.zeros_like(mean)
        return [mean, minv, final, drop]

    out = np.stack(
        stats(ndvi)
        + stats(ndmi)
        + stats(nbr)
        + [valid_count],
        axis=0,
    )

    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return resize_nearest(out, TARGET_SHAPE)


# ============================================================
# S1 FEATURES
# ============================================================
def load_s1_year_features(tile, year, split, ref):
    """
    Fast S1 yearly VV summaries.

    Returns:
      vv_mean, vv_min, vv_max, vv_range, vv_std, valid_count
    Shape: (6, 1000, 1000)
    """
    s1_dir = ROOT / f"sentinel-1/{split}/{tile}__s1_rtc"
    if not s1_dir.exists():
        return np.zeros((6, *TARGET_SHAPE), dtype=np.float32)

    files = sorted(s1_dir.glob(f"*_{year}_*.tif"))
    vv_list = []

    for f in files:
        with rasterio.open(f) as src:
            if src.shape[0] < 100 or src.shape[1] < 100:
                continue

            try:
                vv = src.read(1).astype(np.float32)
            except Exception:
                continue

            out = resize_bilinear_to_ref(vv, src, ref)

        out = np.where(out > 0, out, np.nan)
        vv_list.append(out)

    if len(vv_list) == 0:
        return np.zeros((6, *TARGET_SHAPE), dtype=np.float32)

    vv = np.stack(vv_list)

    with np.errstate(all="ignore"):
        mean = np.nanmean(vv, axis=0)
        minv = np.nanmin(vv, axis=0)
        maxv = np.nanmax(vv, axis=0)
        rangev = maxv - minv
        std = np.nanstd(vv, axis=0)
        valid_count = np.isfinite(vv).sum(axis=0).astype(np.float32)

    out = np.stack([mean, minv, maxv, rangev, std, valid_count], axis=0)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return resize_nearest(out, TARGET_SHAPE)


# ============================================================
# FEATURE EXTRACTION
# ============================================================
def extract_tile_features(tile, split, pca):
    ref = get_ref(tile, split)
    if ref is None:
        return None, None

    feats = []

    # Static AEF PCA.
    feats.append(transform_aef_pca(tile, split, ref, pca))

    # Yearly S2/S1 summaries.
    s2_by_year = []
    s1_by_year = []

    for year in YEARS:
        s2 = load_s2_year_features(tile, year, split, ref)
        s1 = load_s1_year_features(tile, year, split, ref)

        s2_by_year.append(s2)
        s1_by_year.append(s1)

        feats.append(s2)
        feats.append(s1)

    # Simple year-to-year deltas:
    # S2 indices:
    # NDVI final index 2
    # NDMI final index 6
    # NBR final index 10
    # S1 VV mean index 0
    deltas = []
    for i in range(len(YEARS) - 1):
        deltas.append(s2_by_year[i + 1][2] - s2_by_year[i][2])
        deltas.append(s2_by_year[i + 1][6] - s2_by_year[i][6])
        deltas.append(s2_by_year[i + 1][10] - s2_by_year[i][10])
        deltas.append(s1_by_year[i + 1][0] - s1_by_year[i][0])

    feats.append(np.stack(deltas).astype(np.float32))

    all_feats = np.concatenate(feats, axis=0).astype(np.float32)
    X = all_feats.reshape(all_feats.shape[0], -1).T

    return X, ref


def get_feature_names():
    names = []

    for i in range(N_AEF_PCA):
        names.append(f"aef_pca_{i + 1}")
    names.append("aef_valid")

    s2_stats = [
        "ndvi_mean", "ndvi_min", "ndvi_final", "ndvi_drop",
        "ndmi_mean", "ndmi_min", "ndmi_final", "ndmi_drop",
        "nbr_mean", "nbr_min", "nbr_final", "nbr_drop",
        "s2_valid_count",
    ]

    s1_stats = [
        "vv_mean", "vv_min", "vv_max", "vv_range", "vv_std", "s1_valid_count",
    ]

    for year in YEARS:
        for s in s2_stats:
            names.append(f"s2_{year}_{s}")
        for s in s1_stats:
            names.append(f"s1_{year}_{s}")

    for i in range(len(YEARS) - 1):
        y0 = YEARS[i]
        y1 = YEARS[i + 1]
        names.extend([
            f"delta_{y0}_{y1}_ndvi_final",
            f"delta_{y0}_{y1}_ndmi_final",
            f"delta_{y0}_{y1}_nbr_final",
            f"delta_{y0}_{y1}_vv_mean",
        ])

    return names


# ============================================================
# TRAINING SAMPLE
# ============================================================
def sample_training_rows(tile, X, rng):
    p = PGT_DIR / f"{tile}.npz"
    if not p.exists():
        return None

    pgt = np.load(p)
    label = pgt["label"].reshape(-1)
    weight = pgt["confidence"].reshape(-1)

    valid = ~np.isnan(label)
    pos = np.where(valid & (label == 1))[0]
    neg = np.where(valid & (label == 0))[0]

    if len(pos) == 0 or len(neg) == 0:
        return None

    n_pos = min(TRAIN_POS_PER_TILE, len(pos))
    n_neg = min(TRAIN_NEG_PER_TILE, len(neg))

    pos_keep = rng.choice(pos, size=n_pos, replace=False)
    neg_keep = rng.choice(neg, size=n_neg, replace=False)

    keep = np.concatenate([pos_keep, neg_keep])
    rng.shuffle(keep)

    return X[keep], label[keep].astype(np.float32), weight[keep].astype(np.float32)


# ============================================================
# MAIN
# ============================================================
def main():
    rng = np.random.default_rng(RANDOM_SEED)

    train_tiles = sorted([
        p.name.replace("__s2_l2a", "")
        for p in (ROOT / "sentinel-2/train").iterdir()
        if p.is_dir() and p.name.endswith("__s2_l2a")
    ])

    test_tiles = sorted([
        p.name.replace("__s2_l2a", "")
        for p in (ROOT / "sentinel-2/test").iterdir()
        if p.is_dir() and p.name.endswith("__s2_l2a")
    ])

    print(f"Train tiles: {len(train_tiles)}")
    print(f"Test tiles: {len(test_tiles)}")

    pca = fit_aef_pca(train_tiles)

    X_all = []
    y_all = []
    w_all = []

    print("\n=== Extracting train features ===")

    for tile in tqdm(train_tiles, desc="train"):
        X_tile, _ = extract_tile_features(tile, "train", pca)
        if X_tile is None:
            print(f"{tile}: skipped")
            continue

        sampled = sample_training_rows(tile, X_tile, rng)
        if sampled is None:
            print(f"{tile}: no labels")
            continue

        X_s, y_s, w_s = sampled
        X_all.append(X_s)
        y_all.append(y_s)
        w_all.append(w_s)

        print(
            f"{tile}: rows={len(y_s):,}, "
            f"pos={(y_s == 1).sum():,}, neg={(y_s == 0).sum():,}"
        )

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    y = np.concatenate(y_all).astype(np.float32)
    w = np.concatenate(w_all).astype(np.float32)

    del X_all, y_all, w_all

    print("\n=== Train matrix ===")
    print(f"X: {X.shape}, memory={X.nbytes / 1e9:.2f} GB")
    print(f"positive rate: {(y == 1).mean():.2%}")
    print(f"mean weight pos: {w[y == 1].mean():.3f}")
    print(f"mean weight neg: {w[y == 0].mean():.3f}")

    names = get_feature_names()
    print(f"Features: {len(names)}")
    assert X.shape[1] == len(names), (X.shape[1], len(names))

    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "learning_rate": 0.04,
        "num_leaves": 31,
        "max_depth": 6,
        "min_data_in_leaf": 500,
        "feature_fraction": 0.75,
        "bagging_fraction": 0.80,
        "bagging_freq": 1,
        "lambda_l1": 0.2,
        "lambda_l2": 4.0,
        "min_gain_to_split": 0.01,
        "verbose": -1,
        "n_jobs": -1,
        "seed": RANDOM_SEED,
    }

    print("\n=== Training LightGBM ===")

    dtrain = lgb.Dataset(X, y, weight=w, feature_name=names)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=700,
        callbacks=[lgb.log_evaluation(50)],
    )

    model_path = MODEL_DIR / "quick_lgbm_s1_s2_aefpca.txt"
    model.save_model(str(model_path))
    print(f"Saved model: {model_path}")

    importance = pd.DataFrame({
        "feature": names,
        "gain": model.feature_importance(importance_type="gain"),
    }).sort_values("gain", ascending=False)

    importance.to_csv(OUT_DIR / "feature_importance.csv", index=False)
    print("\nTop 30 features:")
    print(importance.head(30).to_string(index=False))

    print("\n=== Predicting test ===")

    all_features_geojson = []

    for tile in tqdm(test_tiles, desc="test"):
        X_tile, ref = extract_tile_features(tile, "test", pca)
        if X_tile is None:
            print(f"{tile}: skipped")
            continue

        pred = model.predict(X_tile).astype(np.float32).reshape(TARGET_SHAPE)

        # Save probability array for cheap threshold experiments later.
        np.save(PRED_DIR / f"{tile}_prob.npy", pred)

        binary = (pred > THRESHOLD).astype(np.uint8)

        h_native, w_native = ref[2]
        h_tgt, w_tgt = TARGET_SHAPE

        rows = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
        cols = (np.arange(w_native) * w_tgt // w_tgt).clip(0, w_tgt - 1)

        prob_native = pred[rows[:, None], cols[None, :]]
        binary_native = binary[rows[:, None], cols[None, :]]

        prob_path = PRED_DIR / f"{tile}_prob.tif"
        pred_path = PRED_DIR / f"{tile}_pred.tif"

        profile_prob = {
            "driver": "GTiff",
            "height": h_native,
            "width": w_native,
            "count": 1,
            "dtype": "float32",
            "crs": ref[1],
            "transform": ref[0],
            "nodata": 0,
            "compress": "deflate",
        }

        with rasterio.open(prob_path, "w", **profile_prob) as dst:
            dst.write(prob_native.astype(np.float32), 1)

        profile_pred = dict(profile_prob)
        profile_pred["dtype"] = "uint8"

        with rasterio.open(pred_path, "w", **profile_pred) as dst:
            dst.write(binary_native.astype(np.uint8), 1)

        geojson = raster_to_geojson(str(pred_path), output_path=None)
        for feat in geojson["features"]:
            feat["properties"]["tile"] = tile

        all_features_geojson.extend(geojson["features"])

        print(
            f"{tile}: pred_px={int(binary_native.sum()):,}, "
            f"polygons={len(geojson['features'])}"
        )

    submission = {
        "type": "FeatureCollection",
        "features": all_features_geojson,
    }

    sub_path = SUB_DIR / "submission_quick_lgbm_s1_s2_aefpca.geojson"
    with open(sub_path, "w") as f:
        json.dump(submission, f)

    print(f"\nSaved submission: {sub_path}")
    print(f"Polygons: {len(all_features_geojson):,}")


if __name__ == "__main__":
    main()


In [15]:
from pathlib import Path
import rasterio

ROOT = Path("data/makeathon-challenge")
PROC = ROOT / "processed"

for d in ["aligned_aef_pca", "s2_cloudmasked_features", "s1_composites"]:
    print("\n", d)
    files = sorted((PROC / d).glob("**/*.tif"))[:10]
    for p in files:
        with rasterio.open(p) as src:
            print(p, src.count, src.shape, src.crs)



 aligned_aef_pca
data/makeathon-challenge/processed/aligned_aef_pca/test/18NVJ_1_6_2020_aef_pca8_on_s2_grid.tif 8 (1002, 1002) EPSG:32618
data/makeathon-challenge/processed/aligned_aef_pca/test/18NVJ_1_6_2021_aef_pca8_on_s2_grid.tif 8 (1002, 1002) EPSG:32618
data/makeathon-challenge/processed/aligned_aef_pca/test/18NVJ_1_6_2022_aef_pca8_on_s2_grid.tif 8 (1002, 1002) EPSG:32618
data/makeathon-challenge/processed/aligned_aef_pca/test/18NVJ_1_6_2023_aef_pca8_on_s2_grid.tif 8 (1002, 1002) EPSG:32618
data/makeathon-challenge/processed/aligned_aef_pca/test/18NVJ_1_6_2024_aef_pca8_on_s2_grid.tif 8 (1002, 1002) EPSG:32618
data/makeathon-challenge/processed/aligned_aef_pca/test/18NVJ_1_6_2025_aef_pca8_on_s2_grid.tif 8 (1002, 1002) EPSG:32618
data/makeathon-challenge/processed/aligned_aef_pca/test/18NYH_2_1_2020_aef_pca8_on_s2_grid.tif 8 (1004, 1004) EPSG:32618
data/makeathon-challenge/processed/aligned_aef_pca/test/18NYH_2_1_2021_aef_pca8_on_s2_grid.tif 8 (1004, 1004) EPSG:32618
data/makeathon

In [16]:
from pathlib import Path

p = Path("data/makeathon-challenge/processed/s2_cloudmasked_features")

print("exists:", p.exists())
print("dirs:")
for d in sorted(p.glob("*"))[:50]:
    print(d)

print("\nfiles:")
for f in sorted(p.glob("**/*"))[:100]:
    if f.is_file():
        print(f)


exists: True
dirs:
data/makeathon-challenge/processed/s2_cloudmasked_features/test
data/makeathon-challenge/processed/s2_cloudmasked_features/train

files:
data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_10_features.npz
data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_11_features.npz
data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_12_features.npz
data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_1_features.npz
data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_2_features.npz
data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_3_features.npz
data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_4_features.npz
d

In [17]:
import numpy as np
from pathlib import Path

p = Path("data/makeathon-challenge/processed/s2_cloudmasked_features/test/18NVJ_1_6__s2_l2a/18NVJ_1_6__s2_l2a_2020_1_features.npz")
z = np.load(p)

print(z.files)
for k in z.files:
    arr = z[k]
    print(k, arr.shape, arr.dtype, np.nanmin(arr), np.nanmax(arr))


['ndvi', 'ndmi', 'nbr', 'b8', 'b11', 'b12', 's2_clear', 'cloud_mask', 'cloud_score']
ndvi (1002, 1002) float32 -0.16462816 0.92126644
ndmi (1002, 1002) float32 -0.29407632 0.71438277
nbr (1002, 1002) float32 -0.2895068 0.8388029
b8 (1002, 1002) float32 0.0413 1.5593
b11 (1002, 1002) float32 0.0257 1.5164
b12 (1002, 1002) float32 0.0123 1.51
s2_clear (1002, 1002) uint8 0 1
cloud_mask (1002, 1002) uint8 0 1
cloud_score (1002, 1002) float32 0.04872683 1.0


In [18]:
def load_s2_cached_year(tile, split, year, target_shape):
    tile_dir = S2_DIR / split / f"{tile}__s2_l2a"
    files = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_*_features.npz"))

    if not files:
        return np.zeros((20, *target_shape), dtype=np.float32)

    series = {
        "ndvi": [],
        "ndmi": [],
        "nbr": [],
    }
    clear_list = []
    cloud_score_list = []

    for p in files:
        z = np.load(p)

        clear = z["s2_clear"].astype(bool)
        clear_list.append(clear.astype(np.float32))
        cloud_score_list.append(z["cloud_score"].astype(np.float32))

        for key in ["ndvi", "ndmi", "nbr"]:
            x = z[key].astype(np.float32)
            x = np.where(clear, x, np.nan)
            series[key].append(x)

    feats = []

    for key in ["ndvi", "ndmi", "nbr"]:
        x = np.stack(series[key], axis=0)

        with np.errstate(all="ignore"):
            mean = np.nanmean(x, axis=0)
            minv = np.nanmin(x, axis=0)
            maxv = np.nanmax(x, axis=0)
            final = x[-1]
            drop = (
                np.nanmin(np.diff(x, axis=0), axis=0)
                if x.shape[0] > 1
                else np.zeros_like(mean)
            )
            valid_count = np.isfinite(x).sum(axis=0).astype(np.float32)

        feats.extend([mean, minv, maxv, final, drop, valid_count])

    clear_stack = np.stack(clear_list, axis=0)
    cloud_stack = np.stack(cloud_score_list, axis=0)

    clear_count = clear_stack.sum(axis=0).astype(np.float32)

    with np.errstate(all="ignore"):
        mean_cloud_score = np.nanmean(cloud_stack, axis=0)

    feats.extend([clear_count, mean_cloud_score])

    out = np.stack(feats).astype(np.float32)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    if out.shape[-2:] != target_shape:
        out = resize_nearest(out, target_shape)

    return out


In [22]:
"""
Full pipeline: all-year AEF features → train LightGBM → predict test → GeoJSON submission
// eas
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import lightgbm as lgb
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import sys, json

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt_codex")
MODEL_DIR = Path("eda_artifacts/model"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR = Path("submission"); SUB_DIR.mkdir(parents=True, exist_ok=True)
TILE_PRED_DIR = Path("eda_artifacts/predictions"); TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]   # all 6 years
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)    # 384 + 5 = 389

NEG_RATIO = 5
N_BOOST_ROUND = 400
CV_HOLDOUT = "18NXH_6_8"   # held-out validation tile


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    """Reference grid for tile (tries S2, then AEF)."""
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best
    # Fallback: AEF
    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape
    return None


def load_aef_to_ref(tile, year, split, ref):
    """Load AEF, reproject all 64 bands to reference grid."""
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"
    if not p.exists(): return None
    
    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)
        for b in range(N_AEF_BANDS):
            reproject(raw[b], aef[b],
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=ref[0], dst_crs=ref[1],
                      resampling=Resampling.bilinear)
    return aef


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape
    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]
    return arr[:, row_idx[:, None], col_idx[None, :]]


def extract_features(tile, split):
    """
    Returns features (N_pixels, N_FEATURES) and reference grid for tile.
    Features: 6×64 AEF values + 5 year-over-year cosine similarities.
    """
    ref = get_ref(tile, split)
    if ref is None: return None, None
    
    # Load all years
    aef_stack = []
    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)
        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None
        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)
    
    # Year-over-year cosine similarities
    changes = []
    for i in range(len(YEARS) - 1):
        a, b = aef_stack[i], aef_stack[i + 1]
        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b
        cos_sim = np.where(denom > 1e-6, dot / (denom + 1e-9), 0.0).astype(np.float32)
        changes.append(cos_sim)
    
    # Stack all features: shape (N_FEATURES, H, W) → (N_pixels, N_FEATURES)
    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0
    )
    features = all_feats.reshape(N_FEATURES, -1).T
    return features, ref


def feature_names():
    names = []
    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")
    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i+1]}")
    return names


# ============================================================
# STEP 1 — Extract features for training
# ============================================================
print(f"=== Step 1: Extract features (target = {N_FEATURES} per pixel) ===")
train_tiles = sorted([p.name.replace("__s2_l2a", "")
                      for p in (ROOT / "sentinel-2/train").iterdir()])

all_feats, all_labels, all_weights, all_tiles = [], [], [], []
for tile in tqdm(train_tiles, desc="train features"):
    pgt_path = PGT_DIR / f"{tile}.npz"
    if not pgt_path.exists(): continue
    
    feats, ref = extract_features(tile, "train")
    if feats is None: continue
    
    pgt = np.load(pgt_path)
    label = pgt["label"].flatten()
    conf = pgt["confidence"].flatten()
    
    # Keep only labeled pixels
    valid = ~np.isnan(label)
    all_feats.append(feats[valid])
    all_labels.append(label[valid])
    all_weights.append(conf[valid])
    all_tiles.append(np.full(int(valid.sum()), tile))

X = np.concatenate(all_feats, axis=0)
y = np.concatenate(all_labels).astype(np.float32)
w = np.concatenate(all_weights).astype(np.float32)
tile_ids = np.concatenate(all_tiles)

print(f"\nTotal labeled pixels: {len(y):,}")
print(f"Positives: {int((y == 1).sum()):,} ({(y == 1).mean():.2%})")
print(f"Negatives: {int((y == 0).sum()):,}")
print(f"Feature matrix: {X.shape}, dtype={X.dtype}")
print(f"Memory: {X.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 2 — Subsample negatives
# ============================================================
print(f"\n=== Step 2: Subsample negatives {NEG_RATIO}x ===")
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_RATIO)
rng = np.random.default_rng(42)
neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)
keep = np.concatenate([pos_idx, neg_sampled])
rng.shuffle(keep)

X_kept = X[keep]
y_kept = y[keep]
w_kept = w[keep]
tiles_kept = tile_ids[keep]

print(f"After subsampling: {len(y_kept):,} rows ({(y_kept == 1).mean():.1%} positive)")
print(f"Memory: {X_kept.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 3 — Held-out validation for threshold
# ============================================================
val_mask = tiles_kept == CV_HOLDOUT
tr_mask = ~val_mask
X_tr = X_kept[tr_mask]; y_tr = y_kept[tr_mask]; w_tr = w_kept[tr_mask]
X_val = X_kept[val_mask]; y_val = y_kept[val_mask]

print(f"\nValidation tile: {CV_HOLDOUT}")
print(f"  Train rows: {len(y_tr):,}, Val rows: {len(y_val):,}")


# ============================================================
# STEP 4 — Train LightGBM
# ============================================================
print("\n=== Step 4: Train LightGBM ===")
names = feature_names()

lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr, feature_name=names)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
}

model = lgb.train(
    params, lgb_train,
    num_boost_round=N_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

model.save_model(str(MODEL_DIR / "lgbm_allyears.txt"))


# ============================================================
# STEP 5 — Tune threshold
# ============================================================
print("\n=== Step 5: Tune threshold ===")
val_pred = model.predict(X_val)
best_thr, best_f1 = 0.5, 0
for thr in np.linspace(0.1, 0.9, 17):
    f1 = f1_score(y_val, val_pred > thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr
print(f"Best threshold: {best_thr:.2f} (F1={best_f1:.3f})")


# ============================================================
# STEP 6 — Predict test tiles
# ============================================================
print("\n=== Step 6: Predict test tiles ===")
test_tiles = sorted([p.name.replace("__s2_l2a", "")
                     for p in (ROOT / "sentinel-2/test").iterdir()])
print(f"Test tiles: {test_tiles}")

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue
    
    pred = model.predict(feats)
    binary = (pred > best_thr).astype(np.uint8).reshape(TARGET_SHAPE)
    
    # Resize binary to native resolution for georeferencing
    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE
    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)
    binary_native = binary[row_idx[:, None], col_idx[None, :]]
    
    # Save GeoTIFF
    tile_path = TILE_PRED_DIR / f"{tile}_pred.tif"
    profile = {
        "driver": "GTiff", "height": h_native, "width": w_native,
        "count": 1, "dtype": "uint8",
        "crs": ref[1], "transform": ref[0], "nodata": 0,
    }
    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)
    
    # Convert to GeoJSON (drops polygons < 0.5 ha inside submission_utils)
    geojson = raster_to_geojson(str(tile_path), output_path=None)
    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile
    all_features_geojson.extend(geojson["features"])
    print(f"  {tile}: {int(binary_native.sum()):,} positive pixels, "
          f"{len(geojson['features'])} polygons")


# ============================================================
# STEP 7 — Save submission
# ============================================================
submission = {"type": "FeatureCollection", "features": all_features_geojson}

sub_path = SUB_DIR / "submission.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\n=== Submission saved: {sub_path} ===")
print(f"Total polygons: {len(all_features_geojson)}")


# ============================================================
# STEP 8 — Feature importance (top 15)
# ============================================================
print("\n=== Feature importance (top 15) ===")
importance = pd.DataFrame({
    "feature": names,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
print(importance.head(15).to_string(index=False))

# Also aggregate by year to see which year's AEF matters most
print("\n=== Importance by year ===")
by_year = {y: 0 for y in YEARS}
change_total = 0
for _, row in importance.iterrows():
    feat = row["feature"]
    if feat.startswith("aef_"):
        year = int(feat.split("_")[1])
        by_year[year] += row["gain"]
    elif feat.startswith("change_"):
        change_total += row["gain"]

for y in YEARS:
    print(f"  AEF {y}: {by_year[y]:.0f}")
print(f"  All changes: {change_total:.0f}")

=== Step 1: Extract features (target = 389 per pixel) ===


train features:   0%|          | 0/16 [00:00<?, ?it/s]


Total labeled pixels: 15,419,204
Positives: 2,324,565 (15.08%)
Negatives: 13,094,639
Feature matrix: (15419204, 389), dtype=float32
Memory: 23.99 GB

=== Step 2: Subsample negatives 5x ===
After subsampling: 13,947,390 rows (16.7% positive)
Memory: 21.70 GB

Validation tile: 18NXH_6_8
  Train rows: 13,133,371, Val rows: 814,019

=== Step 4: Train LightGBM ===
Training until validation scores don't improve for 30 rounds
[50]	train's binary_logloss: 0.116353	val's binary_logloss: 0.10137
[100]	train's binary_logloss: 0.0947515	val's binary_logloss: 0.0691652
[150]	train's binary_logloss: 0.0876765	val's binary_logloss: 0.0626541
[200]	train's binary_logloss: 0.0833028	val's binary_logloss: 0.059736
[250]	train's binary_logloss: 0.080151	val's binary_logloss: 0.0577939
[300]	train's binary_logloss: 0.0776726	val's binary_logloss: 0.0561628
[350]	train's binary_logloss: 0.0756302	val's binary_logloss: 0.0549757
[400]	train's binary_logloss: 0.0739269	val's binary_logloss: 0.0547449
Did no

predicting:   0%|          | 0/5 [00:00<?, ?it/s]

  18NVJ_1_6: 3,193 positive pixels, 8 polygons
  18NYH_2_1: 109,588 positive pixels, 202 polygons
  33NTE_5_1: 65,544 positive pixels, 223 polygons
  47QMA_6_2: 18,718 positive pixels, 97 polygons
  48PWA_0_6: 154,803 positive pixels, 312 polygons

=== Submission saved: submission/submission.geojson ===
Total polygons: 842

=== Feature importance (top 15) ===
         feature         gain
change_2022_2023 1.009768e+07
     aef_2020_22 6.763814e+06
change_2023_2024 5.492729e+06
     aef_2020_44 3.707022e+06
change_2020_2021 3.601727e+06
      aef_2025_5 3.102407e+06
change_2024_2025 2.716207e+06
      aef_2020_5 2.015630e+06
change_2021_2022 1.864243e+06
      aef_2020_0 1.491067e+06
      aef_2024_5 1.326231e+06
     aef_2020_36 1.215753e+06
     aef_2020_51 1.090971e+06
      aef_2025_6 1.089825e+06
     aef_2025_33 7.684343e+05

=== Importance by year ===
  AEF 2020: 26327017
  AEF 2021: 1944837
  AEF 2022: 744766
  AEF 2023: 994082
  AEF 2024: 3463463
  AEF 2025: 9841159
  All chang

In [21]:
# New model from codexx : 

In [23]:
"""
Fast cached baseline without postprocessing:
AEF PCA cache + S2 cloudmasked .npz cache + S1 composite cache + pseudo-GT -> LightGBM -> submissions.

Uses:
  data/makeathon-challenge/processed/aligned_aef_pca
  data/makeathon-challenge/processed/s2_cloudmasked_features
  data/makeathon-challenge/processed/s1_composites
  eda_artifacts/pseudo_gt
"""

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import lightgbm as lgb
from tqdm.auto import tqdm

from submission_utils import raster_to_geojson


# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PROC = ROOT / "processed"

AEF_DIR = PROC / "aligned_aef_pca"
S2_DIR = PROC / "s2_cloudmasked_features"
S1_DIR = PROC / "s1_composites"
PGT_DIR = Path("eda_artifacts/pseudo_gt")

OUT_DIR = Path("eda_artifacts/cached_lgbm_fast")
MODEL_DIR = OUT_DIR / "models"
PRED_DIR = OUT_DIR / "predictions"
SUB_DIR = Path("submission")

for d in [OUT_DIR, MODEL_DIR, PRED_DIR, SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
TARGET_SHAPE = (1000, 1000)

N_AEF_PCA = 8
RANDOM_SEED = 42

MAX_POS_PER_TILE = 180_000
NEG_RATIO = 2
NEG_WEIGHT_MULT = 0.5

N_BOOST_ROUND = 700

THRESHOLDS = [0.10, 0.13, 0.17, 0.21]


# ============================================================
# HELPERS
# ============================================================
def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape

    rows = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    cols = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)

    if arr.ndim == 2:
        return arr[rows[:, None], cols[None, :]]

    return arr[:, rows[:, None], cols[None, :]]


def get_ref_from_aef(tile, split):
    p = AEF_DIR / split / f"{tile}_2020_aef_pca8_on_s2_grid.tif"
    if not p.exists():
        files = sorted((AEF_DIR / split).glob(f"{tile}_*_aef_pca8_on_s2_grid.tif"))
        if not files:
            return None
        p = files[0]

    with rasterio.open(p) as src:
        return src.transform, src.crs, src.shape


def parse_s2_month(path):
    m = re.search(r"_(20\d{2})_(\d{1,2})_features\.npz$", path.name)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def parse_s1_month(path):
    m = re.search(r"_s1_mean_db_(20\d{2})_(\d{1,2})\.tif$", path.name)
    if not m:
        return None, None
    return int(m.group(1)), int(m.group(2))


def reproject_tif_to_ref(path, ref, dtype=np.float32):
    out = np.zeros(ref[2], dtype=dtype)

    with rasterio.open(path) as src:
        reproject(
            rasterio.band(src, 1),
            out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref[0],
            dst_crs=ref[1],
            resampling=Resampling.bilinear,
        )

    return out


# ============================================================
# AEF FEATURES
# ============================================================
def load_aef_year(tile, split, year):
    p = AEF_DIR / split / f"{tile}_{year}_aef_pca8_on_s2_grid.tif"
    if not p.exists():
        return None

    with rasterio.open(p) as src:
        arr = src.read().astype(np.float32)

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    if arr.shape[0] != N_AEF_PCA:
        raise ValueError(f"{p}: expected {N_AEF_PCA} bands, got {arr.shape[0]}")

    return resize_nearest(arr, TARGET_SHAPE)


def load_aef_features(tile, split):
    yearly = []

    for y in YEARS:
        arr = load_aef_year(tile, split, y)
        if arr is None:
            return None
        yearly.append(arr)

    feats = []

    # All yearly PCA bands.
    feats.extend(yearly)

    # Long delta.
    feats.append(yearly[-1] - yearly[0])

    # Year-to-year cosine similarity maps.
    cos_maps = []
    for i in range(len(YEARS) - 1):
        a = yearly[i]
        b = yearly[i + 1]

        dot = (a * b).sum(axis=0)
        na = np.linalg.norm(a, axis=0)
        nb = np.linalg.norm(b, axis=0)

        cos = np.where((na * nb) > 1e-6, dot / (na * nb + 1e-9), 0.0)
        cos_maps.append(cos.astype(np.float32))

    feats.append(np.stack(cos_maps, axis=0))

    return np.concatenate(feats, axis=0).astype(np.float32)


# ============================================================
# S2 FEATURES
# ============================================================
def load_s2_cached_year(tile, split, year):
    tile_dir = S2_DIR / split / f"{tile}__s2_l2a"
    files = sorted(tile_dir.glob(f"{tile}__s2_l2a_{year}_*_features.npz"))

    parsed = []
    for p in files:
        y, m = parse_s2_month(p)
        if y == year and m is not None:
            parsed.append((m, p))

    parsed = sorted(parsed, key=lambda x: x[0])

    if not parsed:
        return np.zeros((20, *TARGET_SHAPE), dtype=np.float32)

    series = {"ndvi": [], "ndmi": [], "nbr": []}
    clear_list = []
    cloud_score_list = []

    for _, p in parsed:
        z = np.load(p)

        clear = z["s2_clear"].astype(bool)
        clear_list.append(clear.astype(np.float32))
        cloud_score_list.append(z["cloud_score"].astype(np.float32))

        for key in ["ndvi", "ndmi", "nbr"]:
            x = z[key].astype(np.float32)
            x = np.where(clear, x, np.nan)
            series[key].append(x)

    feats = []

    for key in ["ndvi", "ndmi", "nbr"]:
        x = np.stack(series[key], axis=0)

        with np.errstate(all="ignore"):
            mean = np.nanmean(x, axis=0)
            minv = np.nanmin(x, axis=0)
            maxv = np.nanmax(x, axis=0)
            final = x[-1]
            drop = (
                np.nanmin(np.diff(x, axis=0), axis=0)
                if x.shape[0] > 1
                else np.zeros_like(mean)
            )
            valid_count = np.isfinite(x).sum(axis=0).astype(np.float32)

        feats.extend([mean, minv, maxv, final, drop, valid_count])

    clear_stack = np.stack(clear_list, axis=0)
    cloud_stack = np.stack(cloud_score_list, axis=0)

    clear_count = clear_stack.sum(axis=0).astype(np.float32)

    with np.errstate(all="ignore"):
        mean_cloud_score = np.nanmean(cloud_stack, axis=0)

    feats.extend([clear_count, mean_cloud_score])

    out = np.stack(feats).astype(np.float32)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    if out.shape[-2:] != TARGET_SHAPE:
        out = resize_nearest(out, TARGET_SHAPE)

    return out


def load_s2_features(tile, split):
    yearly = [load_s2_cached_year(tile, split, y) for y in YEARS]

    feats = []
    feats.extend(yearly)

    # Deltas of final values:
    # layout per year:
    # ndvi final idx = 3
    # ndmi final idx = 9
    # nbr final idx = 15
    deltas = []
    for i in range(len(YEARS) - 1):
        deltas.append(yearly[i + 1][3] - yearly[i][3])
        deltas.append(yearly[i + 1][9] - yearly[i][9])
        deltas.append(yearly[i + 1][15] - yearly[i][15])

    feats.append(np.stack(deltas, axis=0).astype(np.float32))

    return np.concatenate(feats, axis=0).astype(np.float32)


# ============================================================
# S1 FEATURES
# ============================================================
def load_s1_cached_year(tile, year, ref):
    files = sorted(S1_DIR.glob(f"{tile}_s1_mean_db_{year}_*.tif"))
    parsed = []

    for p in files:
        if ".ipynb_checkpoints" in str(p):
            continue
        y, m = parse_s1_month(p)
        if y == year and m is not None:
            parsed.append((m, p))

    parsed = sorted(parsed, key=lambda x: x[0])

    if not parsed:
        return np.zeros((7, *TARGET_SHAPE), dtype=np.float32)

    vv_list = []

    for _, p in parsed:
        arr = reproject_tif_to_ref(p, ref, dtype=np.float32)
        arr = np.where(np.isfinite(arr), arr, np.nan)
        vv_list.append(arr)

    vv = np.stack(vv_list, axis=0)

    with np.errstate(all="ignore"):
        mean = np.nanmean(vv, axis=0)
        minv = np.nanmin(vv, axis=0)
        maxv = np.nanmax(vv, axis=0)
        rangev = maxv - minv
        std = np.nanstd(vv, axis=0)
        final = vv[-1]
        valid_count = np.isfinite(vv).sum(axis=0).astype(np.float32)

    out = np.stack([mean, minv, maxv, rangev, std, final, valid_count], axis=0)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

    return resize_nearest(out.astype(np.float32), TARGET_SHAPE)


def load_s1_features(tile, ref):
    yearly = [load_s1_cached_year(tile, y, ref) for y in YEARS]

    feats = []
    feats.extend(yearly)

    # Mean and final deltas.
    deltas = []
    for i in range(len(YEARS) - 1):
        deltas.append(yearly[i + 1][0] - yearly[i][0])
        deltas.append(yearly[i + 1][5] - yearly[i][5])

    feats.append(np.stack(deltas, axis=0).astype(np.float32))

    return np.concatenate(feats, axis=0).astype(np.float32)


# ============================================================
# FULL TILE FEATURE EXTRACTION
# ============================================================
def extract_tile_features(tile, split):
    ref = get_ref_from_aef(tile, split)
    if ref is None:
        return None, None

    aef = load_aef_features(tile, split)
    if aef is None:
        return None, None

    s2 = load_s2_features(tile, split)
    s1 = load_s1_features(tile, ref)

    all_feats = np.concatenate([aef, s2, s1], axis=0).astype(np.float32)
    X = all_feats.reshape(all_feats.shape[0], -1).T

    return X, ref


def feature_names():
    names = []

    for y in YEARS:
        for b in range(N_AEF_PCA):
            names.append(f"aef_{y}_pca_{b + 1}")

    for b in range(N_AEF_PCA):
        names.append(f"aef_delta_2025_2020_pca_{b + 1}")

    for i in range(len(YEARS) - 1):
        names.append(f"aef_cos_{YEARS[i]}_{YEARS[i + 1]}")

    s2_stats = ["mean", "min", "max", "final", "drop", "valid_count"]

    for y in YEARS:
        for idx in ["ndvi", "ndmi", "nbr"]:
            for stat in s2_stats:
                names.append(f"s2_{y}_{idx}_{stat}")

        names.append(f"s2_{y}_clear_count")
        names.append(f"s2_{y}_mean_cloud_score")

    for i in range(len(YEARS) - 1):
        for idx in ["ndvi", "ndmi", "nbr"]:
            names.append(f"s2_delta_{YEARS[i]}_{YEARS[i + 1]}_{idx}_final")

    s1_stats = ["mean", "min", "max", "range", "std", "final", "valid_count"]

    for y in YEARS:
        for stat in s1_stats:
            names.append(f"s1_{y}_vv_{stat}")

    for i in range(len(YEARS) - 1):
        names.append(f"s1_delta_{YEARS[i]}_{YEARS[i + 1]}_vv_mean")
        names.append(f"s1_delta_{YEARS[i]}_{YEARS[i + 1]}_vv_final")

    return names


# ============================================================
# TRAINING SAMPLE
# ============================================================
def sample_training_rows(tile, X, rng):
    p = PGT_DIR / f"{tile}.npz"
    if not p.exists():
        return None

    z = np.load(p)

    y = z["label"].reshape(-1).astype(np.float32)
    w = z["confidence"].reshape(-1).astype(np.float32)

    if y.shape[0] != X.shape[0]:
        raise ValueError(f"{tile}: label pixels {y.shape[0]} != feature pixels {X.shape[0]}")

    valid = ~np.isnan(y)

    pos_idx = np.where(valid & (y == 1.0))[0]
    neg_idx = np.where(valid & (y == 0.0))[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return None

    n_pos = min(MAX_POS_PER_TILE, len(pos_idx))
    n_neg = min(len(neg_idx), n_pos * NEG_RATIO)

    pos_keep = rng.choice(pos_idx, size=n_pos, replace=False)
    neg_keep = rng.choice(neg_idx, size=n_neg, replace=False)

    keep = np.concatenate([pos_keep, neg_keep])
    rng.shuffle(keep)

    X_s = X[keep]
    y_s = y[keep]
    w_s = w[keep]

    w_s[y_s == 0.0] *= NEG_WEIGHT_MULT

    return X_s, y_s, w_s


# ============================================================
# MAIN
# ============================================================
def main():
    rng = np.random.default_rng(RANDOM_SEED)

    train_tiles = sorted([
        p.name.replace("__s2_l2a", "")
        for p in (ROOT / "sentinel-2/train").iterdir()
        if p.is_dir() and p.name.endswith("__s2_l2a")
    ])

    test_tiles = sorted([
        p.name.replace("__s2_l2a", "")
        for p in (ROOT / "sentinel-2/test").iterdir()
        if p.is_dir() and p.name.endswith("__s2_l2a")
    ])

    print(f"Train tiles: {len(train_tiles)}")
    print(f"Test tiles: {len(test_tiles)}")

    names = feature_names()

    X_parts = []
    y_parts = []
    w_parts = []

    print("\n=== Extracting cached train features ===")

    for tile in tqdm(train_tiles, desc="train"):
        try:
            X_tile, _ = extract_tile_features(tile, "train")

            if X_tile is None:
                print(f"{tile}: skipped, missing features")
                continue

            sampled = sample_training_rows(tile, X_tile, rng)

            if sampled is None:
                print(f"{tile}: skipped, no labels")
                continue

            X_s, y_s, w_s = sampled

            X_parts.append(X_s)
            y_parts.append(y_s)
            w_parts.append(w_s)

            print(
                f"{tile}: rows={len(y_s):,}, "
                f"pos={(y_s == 1).sum():,}, "
                f"neg={(y_s == 0).sum():,}"
            )

        except Exception as e:
            print(f"{tile}: ERROR {e}")

    X = np.concatenate(X_parts, axis=0).astype(np.float32)
    y = np.concatenate(y_parts).astype(np.float32)
    w = np.concatenate(w_parts).astype(np.float32)

    del X_parts, y_parts, w_parts

    print("\n=== Training matrix ===")
    print(f"X: {X.shape}, memory={X.nbytes / 1e9:.2f} GB")
    print(f"features: {len(names)}")
    print(f"positive rate: {(y == 1).mean():.2%}")
    print(f"mean pos weight: {w[y == 1].mean():.3f}")
    print(f"mean neg weight: {w[y == 0].mean():.3f}")

    assert X.shape[1] == len(names), (X.shape[1], len(names))

    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "learning_rate": 0.04,
        "num_leaves": 31,
        "max_depth": 6,
        "min_data_in_leaf": 500,
        "feature_fraction": 0.75,
        "bagging_fraction": 0.80,
        "bagging_freq": 1,
        "lambda_l1": 0.2,
        "lambda_l2": 4.0,
        "min_gain_to_split": 0.01,
        "verbose": -1,
        "n_jobs": -1,
        "seed": RANDOM_SEED,
    }

    print("\n=== Training LightGBM ===")

    dtrain = lgb.Dataset(X, y, weight=w, feature_name=names)

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=N_BOOST_ROUND,
        callbacks=[lgb.log_evaluation(50)],
    )

    model_path = MODEL_DIR / "cached_lgbm_fast_no_post.txt"
    model.save_model(str(model_path))
    print(f"Saved model: {model_path}")

    importance = pd.DataFrame({
        "feature": names,
        "gain": model.feature_importance(importance_type="gain"),
    }).sort_values("gain", ascending=False)

    importance.to_csv(OUT_DIR / "feature_importance_no_post.csv", index=False)

    print("\nTop 30 features:")
    print(importance.head(30).to_string(index=False))

    submissions = {
        f"thr{str(thr).replace('.', '')}": {
            "type": "FeatureCollection",
            "features": [],
        }
        for thr in THRESHOLDS
    }

    print("\n=== Predicting test tiles ===")

    for tile in tqdm(test_tiles, desc="test"):
        try:
            X_tile, ref = extract_tile_features(tile, "test")

            if X_tile is None:
                print(f"{tile}: skipped")
                continue

            prob = model.predict(X_tile).astype(np.float32).reshape(TARGET_SHAPE)
            np.save(PRED_DIR / f"{tile}_prob.npy", prob)

            h_native, w_native = ref[2]
            h_tgt, w_tgt = TARGET_SHAPE

            rows = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
            cols = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)

            prob_native = prob[rows[:, None], cols[None, :]]

            prob_profile = {
                "driver": "GTiff",
                "height": h_native,
                "width": w_native,
                "count": 1,
                "dtype": "float32",
                "crs": ref[1],
                "transform": ref[0],
                "nodata": 0,
                "compress": "deflate",
            }

            prob_tif = PRED_DIR / f"{tile}_prob.tif"

            with rasterio.open(prob_tif, "w", **prob_profile) as dst:
                dst.write(prob_native.astype(np.float32), 1)

            for thr in THRESHOLDS:
                sub_name = f"thr{str(thr).replace('.', '')}"

                binary_native = (prob_native > thr).astype(np.uint8)

                pred_tif = PRED_DIR / f"{tile}_{sub_name}.tif"

                pred_profile = dict(prob_profile)
                pred_profile["dtype"] = "uint8"

                with rasterio.open(pred_tif, "w", **pred_profile) as dst:
                    dst.write(binary_native, 1)

                geojson = raster_to_geojson(str(pred_tif), output_path=None)

                for feat in geojson["features"]:
                    feat["properties"]["tile"] = tile

                submissions[sub_name]["features"].extend(geojson["features"])

                print(
                    f"{tile} {sub_name}: "
                    f"px={int(binary_native.sum()):,}, "
                    f"polygons={len(geojson['features'])}"
                )

        except Exception as e:
            print(f"{tile}: ERROR {e}")

    print("\n=== Saving submissions ===")

    for name, sub in submissions.items():
        path = SUB_DIR / f"submission_cached_lgbm_no_post_{name}.geojson"

        with open(path, "w") as f:
            json.dump(sub, f)

        print(f"{path}: {len(sub['features']):,} polygons")


if __name__ == "__main__":
    main()


Train tiles: 16
Test tiles: 5

=== Extracting cached train features ===


train:   0%|          | 0/16 [00:00<?, ?it/s]

/tmp/ipykernel_6756/2455920613.py:213: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:214: RuntimeWarning: All-NaN slice encountered
  minv = np.nanmin(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:215: RuntimeWarning: All-NaN slice encountered
  maxv = np.nanmax(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:218: RuntimeWarning: All-NaN slice encountered
  np.nanmin(np.diff(x, axis=0), axis=0)


18NWG_6_6: rows=540,000, pos=180,000, neg=360,000


/tmp/ipykernel_6756/2455920613.py:213: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:214: RuntimeWarning: All-NaN slice encountered
  minv = np.nanmin(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:215: RuntimeWarning: All-NaN slice encountered
  maxv = np.nanmax(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:218: RuntimeWarning: All-NaN slice encountered
  np.nanmin(np.diff(x, axis=0), axis=0)
/tmp/ipykernel_6756/2455920613.py:296: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(vv, axis=0)
/tmp/ipykernel_6756/2455920613.py:297: RuntimeWarning: All-NaN slice encountered
  minv = np.nanmin(vv, axis=0)
/tmp/ipykernel_6756/2455920613.py:298: RuntimeWarning: All-NaN slice encountered
  maxv = np.nanmax(vv, axis=0)
/opt/venv/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_6756/2455920613.py:29

...

18NYH_2_1 thr01: px=241,686, polygons=302
18NYH_2_1 thr013: px=216,130, polygons=290
18NYH_2_1 thr017: px=193,938, polygons=267
18NYH_2_1 thr021: px=178,545, polygons=255


/tmp/ipykernel_6756/2455920613.py:213: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:214: RuntimeWarning: All-NaN slice encountered
  minv = np.nanmin(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:215: RuntimeWarning: All-NaN slice encountered
  maxv = np.nanmax(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:218: RuntimeWarning: All-NaN slice encountered
  np.nanmin(np.diff(x, axis=0), axis=0)


33NTE_5_1 thr01: px=364,499, polygons=151
33NTE_5_1 thr013: px=328,561, polygons=103
33NTE_5_1 thr017: px=298,761, polygons=99
33NTE_5_1 thr021: px=277,749, polygons=101


/tmp/ipykernel_6756/2455920613.py:213: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:214: RuntimeWarning: All-NaN slice encountered
  minv = np.nanmin(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:215: RuntimeWarning: All-NaN slice encountered
  maxv = np.nanmax(x, axis=0)
/tmp/ipykernel_6756/2455920613.py:218: RuntimeWarning: All-NaN slice encountered
  np.nanmin(np.diff(x, axis=0), axis=0)


47QMA_6_2 thr01: px=193,628, polygons=499
47QMA_6_2 thr013: px=172,516, polygons=510
47QMA_6_2 thr017: px=151,734, polygons=535
47QMA_6_2 thr021: px=135,732, polygons=524
48PWA_0_6: ERROR all input arrays must have the same shape

=== Saving submissions ===
submission/submission_cached_lgbm_no_post_thr01.geojson: 1,327 polygons
submission/submission_cached_lgbm_no_post_thr013.geojson: 1,202 polygons
submission/submission_cached_lgbm_no_post_thr017.geojson: 1,125 polygons
submission/submission_cached_lgbm_no_post_thr021.geojson: 1,059 polygons


In [24]:
from pathlib import Path
import json

import numpy as np
import rasterio
from submission_utils import raster_to_geojson


PRED_DIR = Path("eda_artifacts/cached_lgbm_fast/predictions")
SUB_DIR = Path("submission")
SUB_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD = 0.15

all_features = []

for prob_path in sorted(PRED_DIR.glob("*_prob.tif")):
    tile = prob_path.name.replace("_prob.tif", "")

    with rasterio.open(prob_path) as src:
        prob = src.read(1).astype(np.float32)
        profile = src.profile.copy()

    binary = (prob > THRESHOLD).astype(np.uint8)

    pred_path = PRED_DIR / f"{tile}_thr015.tif"

    profile.update(
        dtype="uint8",
        count=1,
        nodata=0,
    )

    with rasterio.open(pred_path, "w", **profile) as dst:
        dst.write(binary, 1)

    geojson = raster_to_geojson(str(pred_path), output_path=None)

    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile

    all_features.extend(geojson["features"])

    print(f"{tile}: px={int(binary.sum()):,}, polygons={len(geojson['features'])}")

submission = {
    "type": "FeatureCollection",
    "features": all_features,
}

out_path = SUB_DIR / "submission_cached_lgbm_no_post_thr015.geojson"

with open(out_path, "w") as f:
    json.dump(submission, f)

print(f"\nSaved: {out_path}")
print(f"Polygons: {len(all_features):,}")


18NVJ_1_6: px=89,916, polygons=257
18NYH_2_1: px=203,877, polygons=283
33NTE_5_1: px=311,929, polygons=104
47QMA_6_2: px=161,453, polygons=527

Saved: submission/submission_cached_lgbm_no_post_thr015.geojson
Polygons: 1,171


In [11]:
"""
Full pipeline: all-year AEF features → train LightGBM → predict test → GeoJSON submission
// eas
"""

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import lightgbm as lgb
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import sys, json

sys.path.insert(0, ".")
from submission_utils import raster_to_geojson

# ============================================================
# CONFIG
# ============================================================
ROOT = Path("data/makeathon-challenge")
PGT_DIR = Path("eda_artifacts/pseudo_gt_codex")
MODEL_DIR = Path("eda_artifacts/model"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR = Path("submission"); SUB_DIR.mkdir(parents=True, exist_ok=True)
TILE_PRED_DIR = Path("eda_artifacts/predictions"); TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024, 2025]   # all 6 years
TARGET_SHAPE = (1000, 1000)
N_AEF_BANDS = 64
N_FEATURES = len(YEARS) * N_AEF_BANDS + (len(YEARS) - 1)    # 384 + 5 = 389

NEG_RATIO = 5
N_BOOST_ROUND = 400
CV_HOLDOUT = "18NXH_6_8"   # held-out validation tile


# ============================================================
# HELPERS
# ============================================================
def get_ref(tile, split):
    """Reference grid for tile (tries S2, then AEF)."""
    s2_dir = ROOT / f"sentinel-2/{split}/{tile}__s2_l2a"
    if s2_dir.exists():
        best, best_area = None, 0
        for p in sorted(s2_dir.glob("*.tif")):
            with rasterio.open(p) as r:
                h, w = r.shape
                if h >= 300 and w >= 300 and h * w > best_area:
                    best_area = h * w
                    best = (r.transform, r.crs, r.shape)
        if best is not None:
            return best
    # Fallback: AEF
    aef_files = sorted((ROOT / f"aef-embeddings/{split}").glob(f"{tile}_*.tiff"))
    if aef_files:
        with rasterio.open(aef_files[0]) as r:
            return r.transform, r.crs, r.shape
    return None


def load_aef_to_ref(tile, year, split, ref):
    """Load AEF, reproject all 64 bands to reference grid."""
    p = ROOT / f"aef-embeddings/{split}/{tile}_{year}.tiff"
    if not p.exists(): return None
    
    aef = np.zeros((N_AEF_BANDS, *ref[2]), dtype=np.float32)
    with rasterio.open(p) as src:
        raw = src.read().astype(np.float32)
        raw = np.where(np.isfinite(raw), raw, 0.0)
        for b in range(N_AEF_BANDS):
            reproject(raw[b], aef[b],
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=ref[0], dst_crs=ref[1],
                      resampling=Resampling.bilinear)
    return aef


def resize_nearest(arr, target_shape):
    h_src, w_src = arr.shape[-2:]
    h_tgt, w_tgt = target_shape
    row_idx = (np.arange(h_tgt) * h_src // h_tgt).clip(0, h_src - 1)
    col_idx = (np.arange(w_tgt) * w_src // w_tgt).clip(0, w_src - 1)
    if arr.ndim == 2:
        return arr[row_idx[:, None], col_idx[None, :]]
    return arr[:, row_idx[:, None], col_idx[None, :]]


def extract_features(tile, split):
    """
    Returns features (N_pixels, N_FEATURES) and reference grid for tile.
    Features: 6×64 AEF values + 5 year-over-year cosine similarities.
    """
    ref = get_ref(tile, split)
    if ref is None: return None, None
    
    # Load all years
    aef_stack = []
    for year in YEARS:
        aef = load_aef_to_ref(tile, year, split, ref)
        if aef is None:
            print(f"    Missing AEF for {tile} year {year}")
            return None, None
        aef = resize_nearest(aef, TARGET_SHAPE)
        aef_stack.append(aef)
    
    # Year-over-year cosine similarities
    changes = []
    for i in range(len(YEARS) - 1):
        a, b = aef_stack[i], aef_stack[i + 1]
        dot = (a * b).sum(axis=0)
        norm_a = np.linalg.norm(a, axis=0)
        norm_b = np.linalg.norm(b, axis=0)
        denom = norm_a * norm_b
        cos_sim = np.where(denom > 1e-6, dot / (denom + 1e-9), 0.0).astype(np.float32)
        changes.append(cos_sim)
    
    # Stack all features: shape (N_FEATURES, H, W) → (N_pixels, N_FEATURES)
    all_feats = np.concatenate(
        aef_stack + [c[None] for c in changes],
        axis=0
    )
    features = all_feats.reshape(N_FEATURES, -1).T
    return features, ref


def feature_names():
    names = []
    for y in YEARS:
        for b in range(N_AEF_BANDS):
            names.append(f"aef_{y}_{b}")
    for i in range(len(YEARS) - 1):
        names.append(f"change_{YEARS[i]}_{YEARS[i+1]}")
    return names


# ============================================================
# STEP 1 — Extract features for training
# ============================================================
print(f"=== Step 1: Extract features (target = {N_FEATURES} per pixel) ===")
train_tiles = sorted([p.name.replace("__s2_l2a", "")
                      for p in (ROOT / "sentinel-2/train").iterdir()])

all_feats, all_labels, all_weights, all_tiles = [], [], [], []
for tile in tqdm(train_tiles, desc="train features"):
    pgt_path = PGT_DIR / f"{tile}.npz"
    if not pgt_path.exists(): continue
    
    feats, ref = extract_features(tile, "train")
    if feats is None: continue
    
    pgt = np.load(pgt_path)
    label = pgt["label"].flatten()
    conf = pgt["confidence"].flatten()
    
    # Keep only labeled pixels
    valid = ~np.isnan(label)
    all_feats.append(feats[valid])
    all_labels.append(label[valid])
    all_weights.append(conf[valid])
    all_tiles.append(np.full(int(valid.sum()), tile))

X = np.concatenate(all_feats, axis=0)
y = np.concatenate(all_labels).astype(np.float32)
w = np.concatenate(all_weights).astype(np.float32)
tile_ids = np.concatenate(all_tiles)

print(f"\nTotal labeled pixels: {len(y):,}")
print(f"Positives: {int((y == 1).sum()):,} ({(y == 1).mean():.2%})")
print(f"Negatives: {int((y == 0).sum()):,}")
print(f"Feature matrix: {X.shape}, dtype={X.dtype}")
print(f"Memory: {X.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 2 — Subsample negatives
# ============================================================
print(f"\n=== Step 2: Subsample negatives {NEG_RATIO}x ===")
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_RATIO)
rng = np.random.default_rng(42)
neg_sampled = rng.choice(neg_idx, size=n_neg_keep, replace=False)
keep = np.concatenate([pos_idx, neg_sampled])
rng.shuffle(keep)

X_kept = X[keep]
y_kept = y[keep]
w_kept = w[keep]
tiles_kept = tile_ids[keep]

print(f"After subsampling: {len(y_kept):,} rows ({(y_kept == 1).mean():.1%} positive)")
print(f"Memory: {X_kept.nbytes / 1e9:.2f} GB")


# ============================================================
# STEP 3 — Held-out validation for threshold
# ============================================================
val_mask = tiles_kept == CV_HOLDOUT
tr_mask = ~val_mask
X_tr = X_kept[tr_mask]; y_tr = y_kept[tr_mask]; w_tr = w_kept[tr_mask]
X_val = X_kept[val_mask]; y_val = y_kept[val_mask]

print(f"\nValidation tile: {CV_HOLDOUT}")
print(f"  Train rows: {len(y_tr):,}, Val rows: {len(y_val):,}")


# ============================================================
# STEP 4 — Train LightGBM
# ============================================================
print("\n=== Step 4: Train LightGBM ===")
names = feature_names()

lgb_train = lgb.Dataset(X_tr, y_tr, weight=w_tr, feature_name=names)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.7,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
}

model = lgb.train(
    params, lgb_train,
    num_boost_round=N_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

model.save_model(str(MODEL_DIR / "lgbm_allyears.txt"))


# ============================================================
# STEP 5 — Tune threshold
# ============================================================
print("\n=== Step 5: Tune threshold ===")
val_pred = model.predict(X_val)
best_thr, best_f1 = 0.15, 0

# ============================================================
# STEP 6 — Predict test tiles
# ============================================================
print("\n=== Step 6: Predict test tiles ===")
test_tiles = sorted([p.name.replace("__s2_l2a", "")
                     for p in (ROOT / "sentinel-2/test").iterdir()])
print(f"Test tiles: {test_tiles}")

all_features_geojson = []

for tile in tqdm(test_tiles, desc="predicting"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue
    
    pred = model.predict(feats)
    binary = (pred > best_thr).astype(np.uint8).reshape(TARGET_SHAPE)
    
    # Resize binary to native resolution for georeferencing
    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE
    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)
    binary_native = binary[row_idx[:, None], col_idx[None, :]]
    
    # Save GeoTIFF
    tile_path = TILE_PRED_DIR / f"{tile}_pred.tif"
    profile = {
        "driver": "GTiff", "height": h_native, "width": w_native,
        "count": 1, "dtype": "uint8",
        "crs": ref[1], "transform": ref[0], "nodata": 0,
    }
    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)
    
    # Convert to GeoJSON (drops polygons < 0.5 ha inside submission_utils)
    geojson = raster_to_geojson(str(tile_path), output_path=None)
    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile
    all_features_geojson.extend(geojson["features"])
    print(f"  {tile}: {int(binary_native.sum()):,} positive pixels, "
          f"{len(geojson['features'])} polygons")


# ============================================================
# STEP 7 — Save submission
# ============================================================
submission = {"type": "FeatureCollection", "features": all_features_geojson}

sub_path = SUB_DIR / "submission_first.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\n=== Submission saved: {sub_path} ===")
print(f"Total polygons: {len(all_features_geojson)}")


# ============================================================
# STEP 8 — Feature importance (top 15)
# ============================================================
print("\n=== Feature importance (top 15) ===")
importance = pd.DataFrame({
    "feature": names,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
print(importance.head(15).to_string(index=False))

# Also aggregate by year to see which year's AEF matters most
print("\n=== Importance by year ===")
by_year = {y: 0 for y in YEARS}
change_total = 0
for _, row in importance.iterrows():
    feat = row["feature"]
    if feat.startswith("aef_"):
        year = int(feat.split("_")[1])
        by_year[year] += row["gain"]
    elif feat.startswith("change_"):
        change_total += row["gain"]

for y in YEARS:
    print(f"  AEF {y}: {by_year[y]:.0f}")
print(f"  All changes: {change_total:.0f}")

=== Step 1: Extract features (target = 389 per pixel) ===


train features:   0%|          | 0/16 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
# ============================================================
# FINAL RETRAIN ON ALL TILES + TEST THRESHOLDS 0.10 AND 0.15
# Requires X_kept, y_kept, w_kept, tiles_kept, params, names,
# extract_features, raster_to_geojson from previous cell.
# ============================================================

from pathlib import Path
import json
import numpy as np
import rasterio
import lightgbm as lgb
from tqdm.auto import tqdm

FINAL_NUM_BOOST_ROUND = model.best_iteration
if FINAL_NUM_BOOST_ROUND is None or FINAL_NUM_BOOST_ROUND <= 0:
    FINAL_NUM_BOOST_ROUND = N_BOOST_ROUND

THRESHOLDS = [0.10, 0.15]

print("Retraining on ALL sampled tiles.")
print(f"Rows: {len(y_kept):,}")
print(f"Tiles: {sorted(set(tiles_kept.tolist()))}")
print(f"Using num_boost_round={FINAL_NUM_BOOST_ROUND}")
print(f"Thresholds: {THRESHOLDS}")

lgb_all = lgb.Dataset(
    X_kept,
    y_kept,
    weight=w_kept,
    feature_name=names,
)

final_model = lgb.train(
    params,
    lgb_all,
    num_boost_round=FINAL_NUM_BOOST_ROUND,
    callbacks=[lgb.log_evaluation(50)],
)

final_model_path = MODEL_DIR / "lgbm_allyears_alltiles_from_memory.txt"
final_model.save_model(str(final_model_path))
print(f"Saved final model: {final_model_path}")


test_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/test").iterdir()
    if p.is_dir()
])

submissions = {
    thr: {
        "type": "FeatureCollection",
        "features": [],
    }
    for thr in THRESHOLDS
}

for tile in tqdm(test_tiles, desc="predicting all-tile model"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue

    pred = final_model.predict(feats)
    prob = pred.astype(np.float32).reshape(TARGET_SHAPE)

    np.save(TILE_PRED_DIR / f"{tile}_prob_allyears_alltiles.npy", prob)

    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE

    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)

    prob_native = prob[row_idx[:, None], col_idx[None, :]]

    for thr in THRESHOLDS:
        thr_tag = f"thr{int(round(thr * 100)):03d}"

        binary_native = (prob_native > thr).astype(np.uint8)

        tile_path = TILE_PRED_DIR / f"{tile}_pred_allyears_alltiles_{thr_tag}.tif"

        profile = {
            "driver": "GTiff",
            "height": h_native,
            "width": w_native,
            "count": 1,
            "dtype": "uint8",
            "crs": ref[1],
            "transform": ref[0],
            "nodata": 0,
        }

        with rasterio.open(tile_path, "w", **profile) as dst:
            dst.write(binary_native, 1)

        geojson = raster_to_geojson(str(tile_path), output_path=None)

        for feat in geojson["features"]:
            feat["properties"]["tile"] = tile

        submissions[thr]["features"].extend(geojson["features"])

        print(
            f"  {tile} {thr_tag}: "
            f"{int(binary_native.sum()):,} positive pixels, "
            f"{len(geojson['features'])} polygons"
        )


for thr, submission in submissions.items():
    thr_tag = f"thr{int(round(thr * 100)):03d}"
    sub_path = SUB_DIR / f"submission_allyears_alltiles_{thr_tag}.geojson"

    with open(sub_path, "w") as f:
        json.dump(submission, f)

    print(f"\nSaved submission: {sub_path}")
    print(f"Total polygons: {len(submission['features'])}")


NameError: name 'model' is not defined

In [7]:
# ============================================================
# LOAD SAVED MODEL + RUN TEST WITH THRESHOLD 0.15
# Requires:
# - ROOT, MODEL_DIR, TILE_PRED_DIR, SUB_DIR, TARGET_SHAPE
# - extract_features, raster_to_geojson
# ============================================================

from pathlib import Path
import json
import numpy as np
import rasterio
import lightgbm as lgb
from tqdm.auto import tqdm

THRESHOLDS = [0.15]

final_model_path = MODEL_DIR / "lgbm_allyears_alltiles_from_memory.txt"
print(f"Loading model: {final_model_path}")

final_model = lgb.Booster(model_file=str(final_model_path))

test_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/test").iterdir()
    if p.is_dir()
])

submissions = {
    thr: {
        "type": "FeatureCollection",
        "features": [],
    }
    for thr in THRESHOLDS
}

for tile in tqdm(test_tiles, desc="predicting saved model"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue

    pred = final_model.predict(feats)
    prob = pred.astype(np.float32).reshape(TARGET_SHAPE)

    np.save(TILE_PRED_DIR / f"{tile}_prob_allyears_alltiles.npy", prob)

    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE

    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, w_tgt - 1)

    prob_native = prob[row_idx[:, None], col_idx[None, :]]

    for thr in THRESHOLDS:
        thr_tag = f"thr{int(round(thr * 100)):03d}"

        binary_native = (prob_native > thr).astype(np.uint8)

        tile_path = TILE_PRED_DIR / f"{tile}_pred_allyears_alltiles_{thr_tag}.tif"

        profile = {
            "driver": "GTiff",
            "height": h_native,
            "width": w_native,
            "count": 1,
            "dtype": "uint8",
            "crs": ref[1],
            "transform": ref[0],
            "nodata": 0,
        }

        with rasterio.open(tile_path, "w", **profile) as dst:
            dst.write(binary_native, 1)

        geojson = raster_to_geojson(str(tile_path), output_path=None)

        for feat in geojson["features"]:
            feat["properties"]["tile"] = tile

        submissions[thr]["features"].extend(geojson["features"])

        print(
            f"  {tile} {thr_tag}: "
            f"{int(binary_native.sum()):,} positive pixels, "
            f"{len(geojson['features'])} polygons"
        )

for thr, submission in submissions.items():
    thr_tag = f"thr{int(round(thr * 100)):03d}"
    sub_path = SUB_DIR / f"submission_allyears_alltiles_{thr_tag}.geojson"

    with open(sub_path, "w") as f:
        json.dump(submission, f)

    print(f"\nSaved submission: {sub_path}")
    print(f"Total polygons: {len(submission['features'])}")

NameError: name 'MODEL_DIR' is not defined

In [14]:
# ============================================================
# LOAD SAVED MODEL + RUN TEST WITH THRESHOLD 0.15
# Safe version: skips tiles with zero positive pixels
#
# Assumes these already exist in the notebook:
#   - extract_features(tile, split)
#   - raster_to_geojson(tif_path, output_path=None)
#   - TARGET_SHAPE
# ============================================================

from pathlib import Path
import json
import numpy as np
import rasterio
import lightgbm as lgb
from tqdm.auto import tqdm

# ----------------------------
# Config
# ----------------------------
ROOT = Path("./data/makeathon-challenge")   # change if needed
MODEL_PATH = Path("eda_artifacts/model/lgbm_allyears_alltiles_from_memory.txt")
TILE_PRED_DIR = Path("eda_artifacts/tile_preds")
SUB_DIR = Path("eda_artifacts/submissions")
THR = 0.15
thr_tag = f"thr{int(round(THR * 100)):03d}"

TILE_PRED_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Sanity checks
# ----------------------------
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

if not (ROOT / "sentinel-2/test").exists():
    raise FileNotFoundError(f"Test directory not found: {ROOT / 'sentinel-2/test'}")

missing = []
for name in ["extract_features", "raster_to_geojson", "TARGET_SHAPE"]:
    if name not in globals():
        missing.append(name)

if missing:
    raise NameError(
        "Missing required definitions from previous cells: "
        + ", ".join(missing)
    )

print(f"Loading model: {MODEL_PATH}")
final_model = lgb.Booster(model_file=str(MODEL_PATH))

test_tiles = sorted([
    p.name.replace("__s2_l2a", "")
    for p in (ROOT / "sentinel-2/test").iterdir()
    if p.is_dir()
])

print(f"Found {len(test_tiles)} test tiles")
print(f"Using threshold: {THR}")
print(f"TARGET_SHAPE: {TARGET_SHAPE}")

submission = {
    "type": "FeatureCollection",
    "features": [],
}

empty_tiles = []

for tile in tqdm(test_tiles, desc="predicting saved model"):
    feats, ref = extract_features(tile, "test")
    if feats is None:
        print(f"  {tile}: skipped")
        continue

    pred = final_model.predict(feats)
    prob = pred.astype(np.float32).reshape(TARGET_SHAPE)

    np.save(TILE_PRED_DIR / f"{tile}_prob_allyears_alltiles.npy", prob)

    h_native, w_native = ref[2]
    h_tgt, w_tgt = TARGET_SHAPE

    row_idx = (np.arange(h_native) * h_tgt // h_native).clip(0, h_tgt - 1)
    col_idx = (np.arange(w_native) * w_tgt // w_native).clip(0, h_tgt - 1)

    prob_native = prob[row_idx[:, None], col_idx[None, :]]
    binary_native = (prob_native > THR).astype(np.uint8)

    n_pos = int(binary_native.sum())

    tile_path = TILE_PRED_DIR / f"{tile}_pred_allyears_alltiles_{thr_tag}.tif"

    profile = {
        "driver": "GTiff",
        "height": h_native,
        "width": w_native,
        "count": 1,
        "dtype": "uint8",
        "crs": ref[1],
        "transform": ref[0],
        "nodata": 0,
    }

    with rasterio.open(tile_path, "w", **profile) as dst:
        dst.write(binary_native, 1)

    if n_pos == 0:
        empty_tiles.append(tile)
        print(f"  {tile} {thr_tag}: 0 positive pixels, skipped geojson")
        continue

    geojson = raster_to_geojson(str(tile_path), output_path=None)

    for feat in geojson["features"]:
        feat["properties"]["tile"] = tile

    submission["features"].extend(geojson["features"])

    print(
        f"  {tile} {thr_tag}: "
        f"{n_pos:,} positive pixels, "
        f"{len(geojson['features'])} polygons"
    )

sub_path = SUB_DIR / f"submission_allyears_alltiles_{thr_tag}.geojson"
with open(sub_path, "w") as f:
    json.dump(submission, f)

print(f"\nSaved submission: {sub_path}")
print(f"Total polygons: {len(submission['features'])}")
print(f"Empty tiles skipped: {len(empty_tiles)}")
if empty_tiles:
    print("Tiles with no predicted positives:")
    print(empty_tiles)

Loading model: eda_artifacts/model/lgbm_allyears_alltiles_from_memory.txt
Found 5 test tiles
Using threshold: 0.15
TARGET_SHAPE: (1000, 1000)


predicting saved model:   0%|          | 0/5 [00:00<?, ?it/s]

  18NVJ_1_6 thr015: 10,556 positive pixels, 42 polygons
  18NYH_2_1 thr015: 131,476 positive pixels, 215 polygons
  33NTE_5_1 thr015: 83,873 positive pixels, 247 polygons
  47QMA_6_2 thr015: 0 positive pixels, skipped geojson
  48PWA_0_6 thr015: 193,186 positive pixels, 310 polygons

Saved submission: eda_artifacts/submissions/submission_allyears_alltiles_thr015.geojson
Total polygons: 814
Empty tiles skipped: 1
Tiles with no predicted positives:
['47QMA_6_2']
